# 📊 Relatório de Indicadores

## Como usar:
1. Clique em **Run → Run All Cells**
2. Vá até o final da página
3. Preencha os filtros
4. Clique em **Rodar indicadores**

# Código

## Configuração Inicial 

### Instalação de Bibliotecas

In [195]:
import datetime
import ipywidgets as widgets
from IPython.display import display
import os
from pathlib import Path
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import numpy as np
from scipy.stats import linregress
from sqlalchemy import create_engine
from plotly.subplots import make_subplots

### Ligação com o Banco

In [196]:
%%capture
user = "greg"
password = "CZitU+tEEFoSUy+esV3Gbw=="
host = "172.16.2.114"
port = "9030"
database = "starrocks"

In [197]:
engine = create_engine(
    f"{database}://{user}:{password}@{host}:{port}"
)

query_empresas_unicas = """
WITH pessoas_dim AS (
  SELECT
    p.id AS id_pessoa,
    TRIM(p.nome_organizacao) AS empresa_pessoa
  FROM crm.pessoas p
  WHERE p.id IS NOT NULL
    AND (
      p.nome IS NULL
      OR LOWER(TRIM(p.nome)) NOT LIKE '%%test%%'
    )
    AND (
      p.nome_social IS NULL
      OR LOWER(TRIM(p.nome_social)) NOT LIKE '%%test%%'
    )
    AND p.nome_organizacao IS NOT NULL
    AND TRIM(p.nome_organizacao) <> ''
)

SELECT DISTINCT
  CURRENT_TIMESTAMP AS data_atualizacao,
  empresa_pessoa
FROM pessoas_dim
ORDER BY empresa_pessoa;
"""

df_empresas = pd.read_sql(query_empresas_unicas, engine)

Failed to get run_mode: (pymysql.err.OperationalError) (5203, 'Access denied; you need (at least one of) the OPERATE privilege(s) on SYSTEM for this operation. Please ask the admin to grant permission(s) or try activating existing roles using <set [default] role>. Current role(s): NONE. Inactivated role(s): NONE.')
[SQL: ADMIN SHOW FRONTEND CONFIG LIKE 'run_mode']
(Background on this error at: https://sqlalche.me/e/20/e3q8)


### Criar seleção

In [198]:
# =========================
# Variáveis globais de filtro
# =========================
empresa_selecionada = None
data_inicial_relatorio = None
data_final_relatorio = None
detalhe_relatorio = "completo"

# =========================
# Lista de empresas
# =========================
lista_empresas = sorted(df_empresas["empresa_pessoa"].dropna().unique().tolist())
opcoes_empresas = ["Todos"] + lista_empresas

# =========================
# Widgets do formulário
# =========================
dropdown_empresa = widgets.Dropdown(
    options=opcoes_empresas,
    value="Todos",
    description="Empresa:",
    layout=widgets.Layout(width="50%")
)

data_inicial_widget = widgets.DatePicker(
    description="Data inicial:",
    value=None
)

data_final_widget = widgets.DatePicker(
    description="Data final:",
    value=None
)

dropdown_detalhe = widgets.Dropdown(
    options=[
        ("Completo", "completo"),
        ("Resumido", "nenhum"),
    ],
    value="completo",
    description="Detalhe:",
    layout=widgets.Layout(width="50%")
)

botao_confirmar = widgets.Button(
    description="Confirmar seleção",
    button_style="success"
)

botao_rodar_indicadores = widgets.Button(
    description="Rodar indicadores",
    button_style="primary"
)

saida_filtros = widgets.Output()
saida_execucao = widgets.Output()


In [199]:
# =========================
# Função auxiliar para exibição
# =========================
def formatar_data_br(data):
    if data is None:
        return "Não definida"
    return data.strftime("%d/%m/%Y")

# =========================
# Callback do botão
# =========================
def confirmar_selecao(b):
    global empresa_selecionada, data_inicial_relatorio, data_final_relatorio, detalhe_relatorio

    empresa_selecionada = dropdown_empresa.value
    data_inicial_relatorio = data_inicial_widget.value
    data_final_relatorio = data_final_widget.value
    detalhe_relatorio = dropdown_detalhe.value

    with saida_filtros:
        saida_filtros.clear_output()

        print(f"Empresa selecionada: {empresa_selecionada}")
        print(f"Data inicial: {formatar_data_br(data_inicial_relatorio)}")
        print(f"Data final: {formatar_data_br(data_final_relatorio)}")
        print(f"Detalhe: {detalhe_relatorio}")

In [200]:
def rodar_indicadores(b):
    global resultados_indicadores

    with saida_execucao:
        saida_execucao.clear_output()

        print("Iniciando execução dos indicadores...")
        print(f"Empresa: {empresa_selecionada}")
        print(f"Data inicial: {formatar_data_br(data_inicial_relatorio)}")
        print(f"Data final: {formatar_data_br(data_final_relatorio)}")
        print(f"Detalhe: {detalhe_relatorio}")
        print("")

        resultados_indicadores = executar_todos_indicadores(
            engine=engine,
            nome_pasta="saida_indicadores",
            detalhe=detalhe_relatorio
        )

        print("")
        print("Execução finalizada.")

In [201]:
botao_confirmar.on_click(confirmar_selecao)
botao_rodar_indicadores.on_click(rodar_indicadores)

## Utilirários gerais

In [202]:
def garantir_pasta_saida(nome_pasta="saida_indicadores"):
    pasta = Path(nome_pasta)
    pasta.mkdir(parents=True, exist_ok=True)
    return pasta

def nome_filtro_empresa():
    if empresa_selecionada is None or empresa_selecionada == "Todos":
        return "todas_empresas"

    nome = str(empresa_selecionada).lower().strip()
    nome = nome.replace(" ", "_").replace("/", "_").replace("\\", "_")
    nome = nome.replace("'", "").replace('"', "")
    return nome

def nome_periodo():
    dt_ini = data_inicial_relatorio.strftime("%Y%m%d") if data_inicial_relatorio else "sem_data_ini"
    dt_fim = data_final_relatorio.strftime("%Y%m%d") if data_final_relatorio else "sem_data_fim"
    return f"{dt_ini}_a_{dt_fim}"

def executar_query(query, engine):
    return pd.read_sql(query, engine)

def prefixo_ordem_indicador(nome_indicador):
    if nome_indicador not in catalogo_indicadores:
        return "00"
    
    ordem = catalogo_indicadores[nome_indicador].get("ordem", 0)
    return f"{ordem:02d}"

def exportar_dataframe(df, nome_indicador, nome_pasta="saida_indicadores", tipo_df="dados"):
    pasta = garantir_pasta_saida(nome_pasta)
    prefixo = prefixo_ordem_indicador(nome_indicador)
    nome_arquivo = f"{prefixo}_{nome_indicador}_{tipo_df}_{nome_filtro_empresa()}_{nome_periodo()}.csv"
    caminho = pasta / nome_arquivo
    df.to_csv(caminho, index=False, encoding="utf-8-sig")
    print(f"CSV exportado: {caminho}")
    return caminho

def exportar_grafico_html(fig, nome_indicador, nome_pasta="saida_indicadores"):
    pasta = garantir_pasta_saida(nome_pasta)
    prefixo = prefixo_ordem_indicador(nome_indicador)
    nome_arquivo = f"{prefixo}_{nome_indicador}_grafico_{nome_filtro_empresa()}_{nome_periodo()}.html"
    caminho = pasta / nome_arquivo
    fig.write_html(caminho)
    print(f"Gráfico HTML exportado: {caminho}")
    return caminho
    
def exportar_grafico_png(fig, nome_indicador, nome_pasta="saida_indicadores"):
    pasta = garantir_pasta_saida(nome_pasta)
    prefixo = prefixo_ordem_indicador(nome_indicador)
    nome_arquivo = f"{prefixo}_{nome_indicador}_grafico_{nome_filtro_empresa()}_{nome_periodo()}.png"
    caminho = pasta / nome_arquivo
    fig.write_image(str(caminho), width=2000, height=800, scale=1)
    print(f"PNG exportado: {caminho}")
    return caminho

In [203]:
print(nome_filtro_empresa())
print(nome_periodo())

pasta_teste = garantir_pasta_saida()
print(pasta_teste)

todas_empresas
sem_data_ini_a_sem_data_fim
saida_indicadores


## Utilirários de gráficos

### Grafico_linha_temporal_completo

In [204]:
def grafico_linha_temporal_completo(df, x, y, titulo, detalhe="completo"):
    serie = df.copy()

    # Garantir datetime
    serie[x] = pd.to_datetime(serie[x], errors="coerce")

    # Limpar e ordenar
    serie = (
        serie.dropna(subset=[x, y])
        .sort_values(by=x)
        .reset_index(drop=True)
        .copy()
    )

    serie["indice"] = range(len(serie))

    # Métricas
    media = serie[y].mean()
    serie["variacao_pop"] = serie[y].pct_change() * 100
    serie["variacao_yoy"] = serie[y].pct_change(periods=12) * 100

    # Tendência
    if len(serie) >= 2:
        slope, intercept, *_ = linregress(serie["indice"], serie[y])
        serie["tendencia"] = intercept + slope * serie["indice"]
    else:
        serie["tendencia"] = np.nan

    # Texto dos valores
    serie["texto_valor"] = serie[y].apply(
        lambda v: f"{v:,.0f}".replace(",", ".")
    )

    # Amplitude do eixo para usar nos deslocamentos
    y_min = serie[y].min()
    y_max = serie[y].max()
    amplitude_y = max(y_max - y_min, 1)

    def obter_posicao_anotacao(i, serie, amplitude_y):
        y_atual = serie.loc[i, y]

        y_ant = serie.loc[i - 1, y] if i > 0 else np.nan
        y_prox = serie.loc[i + 1, y] if i < len(serie) - 1 else np.nan

        eh_pico_local = (
            pd.notna(y_ant) and pd.notna(y_prox) and
            y_atual >= y_ant and y_atual >= y_prox
        )

        eh_vale_local = (
            pd.notna(y_ant) and pd.notna(y_prox) and
            y_atual <= y_ant and y_atual <= y_prox
        )

        if eh_pico_local:
            yshift = 38
        elif eh_vale_local:
            yshift = -38
        else:
            pop = serie.loc[i, "variacao_pop"]
            if pd.notna(pop) and pop < 0:
                yshift = -34
            else:
                yshift = 34

        distancia_topo = (y_max - y_atual) / amplitude_y
        if distancia_topo < 0.12:
            yshift = -42

        distancia_base = (y_atual - y_min) / amplitude_y
        if distancia_base < 0.12:
            yshift = 42

        return yshift

    detalhes_validos = {"completo", "pop", "yoy", "nenhum"}
    if detalhe not in detalhes_validos:
        raise ValueError(
            f"Parâmetro 'detalhe' inválido. Use um destes: {sorted(detalhes_validos)}"
        )

    fig = go.Figure()

    # Série principal
    fig.add_trace(go.Scatter(
        x=serie[x],
        y=serie[y],
        mode="lines+markers+text",
        name="Valor",
        text=serie["texto_valor"],
        textposition="top center",
        line=dict(width=3),
        marker=dict(size=8)
    ))

    # Tendência
    if len(serie) >= 2:
        fig.add_trace(go.Scatter(
            x=serie[x],
            y=serie["tendencia"],
            mode="lines",
            name="Tendência",
            line=dict(dash="dash", width=2)
        ))

    # Anotações PoP e YoY
    if detalhe != "nenhum":
        for i, row in serie.iterrows():
            pop = row["variacao_pop"]
            yoy = row["variacao_yoy"]

            textos = []
            cores = []

            if detalhe in {"completo", "pop"} and pd.notna(pop):
                pop_int = int(round(abs(pop), 0))
                seta_pop = "▲" if pop > 0 else "▼"
                cor_pop = "green" if pop > 0 else "red"
                textos.append(f"{seta_pop} {pop_int}% PoP")
                cores.append(cor_pop)

            if detalhe in {"completo", "yoy"} and pd.notna(yoy):
                yoy_int = int(round(abs(yoy), 0))
                seta_yoy = "▲" if yoy > 0 else "▼"
                cor_yoy = "green" if yoy > 0 else "red"
                textos.append(f"{seta_yoy} {yoy_int}% YoY")
                cores.append(cor_yoy)

            if textos:
                if len(textos) == 1:
                    texto_html = f"<span style='color:{cores[0]}'>{textos[0]}</span>"
                else:
                    texto_html = (
                        f"<span style='color:{cores[0]}'>{textos[0]}</span><br>"
                        f"<span style='color:{cores[1]}'>{textos[1]}</span>"
                    )

                yshift = obter_posicao_anotacao(i, serie, amplitude_y)

                fig.add_annotation(
                    x=row[x],
                    y=row[y],
                    text=texto_html,
                    showarrow=False,
                    font=dict(size=10),
                    yshift=yshift,
                    xanchor="center",
                    align="center"
                )

    # Linha de média
    if pd.notna(media):
        fig.add_hline(
            y=media,
            line_dash="dash",
            line_color="gray"
        )

        media_txt = f"{media:,.0f}".replace(",", "X").replace(".", ",").replace("X", ".")

        fig.add_annotation(
            x=1.01,
            xref="paper",
            y=media,
            yref="y",
            text=f"Média: {media_txt}",
            showarrow=False,
            font=dict(size=11),
            align="left",
            bgcolor="rgba(255,255,255,0.7)"
        )

    # Layout
    fig.update_layout(
        title={
            "text": f"<b>{titulo}</b>",
            "x": 0.5,
            "xanchor": "center"
        },
        xaxis_title="",
        yaxis_title="",
        width=2000,
        height=800,
        paper_bgcolor="white",
        plot_bgcolor="white",
        margin=dict(l=40, r=40, t=80, b=80),
        legend=dict(
            orientation="h",
            yanchor="bottom",
            y=1.02,
            xanchor="right",
            x=1
        )
    )

    fig.update_xaxes(
        type="date",
        tickformat="%Y-%m",
        tickangle=0,
        showgrid=False,
        showline=False,
        zeroline=False
    )

    fig.update_yaxes(
        showgrid=False,
        showline=False,
        zeroline=False
    )

    return fig

In [205]:
df_teste = pd.DataFrame({
    "competencia": [
        "2024-01","2024-02","2024-03","2024-04","2024-05","2024-06",
        "2024-07","2024-08","2024-09","2024-10","2024-11","2024-12",
        "2025-01","2025-02","2025-03","2025-04","2025-05","2025-06"
    ],
    "valor": [
        90, 95, 100, 105, 110, 108,
        112, 115, 118, 120, 122, 125,
        100, 120, 115, 140, 135, 150
    ]
})

fig1 = grafico_linha_temporal_completo(
    df=df_teste,
    x="competencia",
    y="valor",
    titulo="Teste gráfico completo com PoP e YoY",
    detalhe="nenhum"
)

fig1.show()

### Grafico_dinamica_entradas_saidas

In [206]:
def grafico_dinamica_entrada_saida(
    turnover_plot,
    detalhe="completo",
    titulo="Dinâmica mensal de beneficiários – Entradas e Saídas"
):
    df = turnover_plot.copy().reset_index(drop=True)

    detalhes_validos = {"completo", "pop", "yoy", "nenhum"}
    if detalhe not in detalhes_validos:
        raise ValueError(
            f"Parâmetro 'detalhe' inválido. Use um destes: {sorted(detalhes_validos)}"
        )

    # =========================
    # 1) Métricas auxiliares
    # =========================
    df["pop_entradas"] = df["entradas"].pct_change() * 100
    df["pop_saidas"] = df["saidas"].pct_change() * 100

    df["yoy_entradas"] = df["entradas"].pct_change(periods=12) * 100
    df["yoy_saidas"] = df["saidas"].pct_change(periods=12) * 100

    media_entradas = df["entradas"].mean()
    media_saidas = df["saidas"].mean()

    df["texto_entradas"] = df["entradas"].apply(lambda v: f"{v:,.0f}".replace(",", "."))
    df["texto_saidas"] = df["saidas"].apply(lambda v: f"-{v:,.0f}".replace(",", "."))

    fig = go.Figure()

    # =========================
    # 2) Barras - Entradas
    # =========================
    fig.add_trace(
        go.Bar(
            x=df["competencia"],
            y=df["entradas"],
            name="Entradas",
            marker_color="#2ca02c",
            text=df["texto_entradas"],
            textposition="inside",
            textfont=dict(color="white", size=11),
            insidetextanchor="middle"
        )
    )

    # =========================
    # 3) Barras - Saídas
    # =========================
    fig.add_trace(
        go.Bar(
            x=df["competencia"],
            y=-df["saidas"],
            name="Saídas",
            marker_color="#d62728",
            text=df["texto_saidas"],
            textposition="inside",
            textfont=dict(color="white", size=11),
            insidetextanchor="middle"
        )
    )

    # =========================
    # 4) Anotações - Entradas
    # =========================
    if detalhe != "nenhum":
        for _, row in df.iterrows():
            textos = []

            if detalhe in {"completo", "pop"} and pd.notna(row["pop_entradas"]):
                seta = "▲" if row["pop_entradas"] > 0 else "▼"
                textos.append(f"PoP {seta} {abs(row['pop_entradas']):.0f}%")

            if detalhe in {"completo", "yoy"} and pd.notna(row["yoy_entradas"]):
                seta = "▲" if row["yoy_entradas"] > 0 else "▼"
                textos.append(f"YoY {seta} {abs(row['yoy_entradas']):.0f}%")

            if textos:
                fig.add_annotation(
                    x=row["competencia"],
                    y=row["entradas"],
                    text="<br>".join(textos),
                    showarrow=False,
                    font=dict(size=10, color="green"),
                    yshift=28
                )

    # =========================
    # 5) Anotações - Saídas
    # =========================
    if detalhe != "nenhum":
        for _, row in df.iterrows():
            textos = []

            if detalhe in {"completo", "pop"} and pd.notna(row["pop_saidas"]):
                seta = "▲" if row["pop_saidas"] > 0 else "▼"
                textos.append(f"PoP {seta} {abs(row['pop_saidas']):.0f}%")

            if detalhe in {"completo", "yoy"} and pd.notna(row["yoy_saidas"]):
                seta = "▲" if row["yoy_saidas"] > 0 else "▼"
                textos.append(f"YoY {seta} {abs(row['yoy_saidas']):.0f}%")

            if textos:
                fig.add_annotation(
                    x=row["competencia"],
                    y=-row["saidas"],
                    text="<br>".join(textos),
                    showarrow=False,
                    font=dict(size=10, color="red"),
                    yshift=-28
                )

    # =========================
    # 6) Linhas médias
    # =========================
    fig.add_hline(
        y=media_entradas,
        line_dash="dash",
        line_color="#2ca02c"
    )

    fig.add_hline(
        y=-media_saidas,
        line_dash="dash",
        line_color="#d62728"
    )

    fig.add_annotation(
        x=1.02,
        xref="paper",
        y=media_entradas,
        yref="y",
        text=f"Média: {media_entradas:,.0f}".replace(",", "X").replace(".", ",").replace("X", "."),
        showarrow=False,
        font=dict(size=11, color="#2ca02c"),
        align="left",
        xanchor="left",
        yanchor="middle",
        bgcolor="rgba(255,255,255,0.8)"
    )

    fig.add_annotation(
        x=1.02,
        xref="paper",
        y=-media_saidas,
        yref="y",
        text=f"Média: -{media_saidas:,.0f}".replace(",", "X").replace(".", ",").replace("X", "."),
        showarrow=False,
        font=dict(size=11, color="#d62728"),
        align="left",
        xanchor="left",
        yanchor="middle",
        bgcolor="rgba(255,255,255,0.8)"
    )

    # =========================
    # 7) Layout final
    # =========================
    fig.update_layout(
        title=titulo,
        barmode="relative",
        xaxis=dict(title="Competência"),
        yaxis=dict(
            title="Entradas / Saídas",
            zeroline=True,
            zerolinewidth=2,
            zerolinecolor="gray"
        ),
        legend=dict(
            orientation="h",
            yanchor="bottom",
            y=1.05,
            xanchor="right",
            x=1
        ),
        template="plotly_white",
        width=2400,
        height=800,
        margin=dict(l=40, r=140, t=80, b=80)
    )

    return fig

In [207]:
df = pd.DataFrame({
    "competencia": [
        "2024-01","2024-02","2024-03","2024-04","2024-05","2024-06",
        "2024-07","2024-08","2024-09","2024-10","2024-11","2024-12",
        "2025-01","2025-02"
    ],
    "entradas": [50, 60, 55, 70, 65, 80, 75, 68, 72, 78, 74, 82, 58, 66],
    "saidas":   [30, 25, 40, 35, 50, 45, 42, 38, 36, 41, 39, 44, 34, 29],
    "ativos":   [500, 535, 550, 585, 600, 635, 668, 698, 734, 771, 806, 844, 868, 905]
})

fig1 = grafico_dinamica_entrada_saida(
    df,
    detalhe="completo",
    titulo="Entradas e saídas – análise operacional"
)

fig1.show()

### Grafico_piramide_etaria

In [208]:
def grafico_piramide_etaria(
    df_plot,
    titulo="Pirâmide etária por sexo - último mês",
    incluir_nao_informado=True
):
    base = df_plot.copy()

    masculino = base[base["genero"] == "Masculino"].copy()
    feminino = base[base["genero"] == "Feminino"].copy()
    nao_inf = base[base["genero"] == "Não informado"].copy()

    masculino["valor_plot"] = -masculino["qtd_ativos"]
    feminino["valor_plot"] = feminino["qtd_ativos"]
    nao_inf["valor_plot"] = nao_inf["qtd_ativos"]

    fig = go.Figure()

    fig.add_trace(go.Bar(
        y=masculino["faixa_etaria"],
        x=masculino["valor_plot"],
        name="Masculino",
        orientation="h",
        marker_color="rgb(93, 135, 222)",
        text=masculino["qtd_ativos"].astype(int),
        textposition="auto"
    ))

    fig.add_trace(go.Bar(
        y=feminino["faixa_etaria"],
        x=feminino["valor_plot"],
        name="Feminino",
        orientation="h",
        marker_color="rgb(226, 115, 145)",
        text=feminino["qtd_ativos"].astype(int),
        textposition="auto"
    ))

    if incluir_nao_informado and not nao_inf["qtd_ativos"].eq(0).all():
        fig.add_trace(go.Bar(
            y=nao_inf["faixa_etaria"],
            x=nao_inf["valor_plot"],
            name="Não informado",
            orientation="h",
            marker_color="#9D9D9D",
            text=nao_inf["qtd_ativos"].astype(int),
            textposition="auto"
        ))

    max_abs = max(
        abs(masculino["valor_plot"]).max() if not masculino.empty else 0,
        feminino["valor_plot"].max() if not feminino.empty else 0,
        nao_inf["valor_plot"].max() if not nao_inf.empty else 0
    )

    fig.update_layout(
        title=titulo,
        barmode="relative",
        template="plotly_white",
        width=1400,
        height=800,
        bargap=0.15,
        legend=dict(
            orientation="h",
            yanchor="bottom",
            y=1.02,
            xanchor="right",
            x=1
        ),
        xaxis=dict(
            title="Quantidade de beneficiários ativos",
            tickvals=[-max_abs, -max_abs/2, 0, max_abs/2, max_abs],
            ticktext=[f"{int(max_abs)}", f"{int(max_abs/2)}", "0", f"{int(max_abs/2)}", f"{int(max_abs)}"]
        ),
        yaxis=dict(
            title="Faixa etária",
            categoryorder="array",
            categoryarray=list(base["faixa_etaria"].drop_duplicates())
        ),
        margin=dict(l=80, r=80, t=80, b=80)
    )

    fig.add_vline(x=0, line_width=1, line_color="gray")

    return fig

In [209]:
df_teste_piramide = pd.DataFrame({
    "faixa_etaria": [
        "0-17", "0-17",
        "18-24", "18-24",
        "25-34", "25-34",
        "35-44", "35-44",
        "45-54", "45-54",
        "55-64", "55-64",
        "65-74", "65-74",
        "75+", "75+"
    ],
    "genero": [
        "Masculino", "Feminino",
        "Masculino", "Feminino",
        "Masculino", "Feminino",
        "Masculino", "Feminino",
        "Masculino", "Feminino",
        "Masculino", "Feminino",
        "Masculino", "Feminino",
        "Masculino", "Feminino"
    ],
    "qtd_ativos": [
        18, 20,
        25, 28,
        40, 45,
        38, 42,
        30, 35,
        22, 26,
        15, 18,
        8, 10
    ]
})

fig1 = grafico_piramide_etaria(
    df_plot=df_teste_piramide,
    titulo="Pirâmide etária - teste manual"
)

fig1.show()

### Grafico_cartao_numerico

In [210]:
def grafico_card_ativos_ultimo_mes(
    df_plot,
    detalhe="completo",
    titulo="Total de beneficiários ativos - última competência"
):
    detalhes_validos = {"completo", "pop", "yoy", "nenhum"}
    if detalhe not in detalhes_validos:
        raise ValueError(
            f"Parâmetro 'detalhe' inválido. Use um destes: {sorted(detalhes_validos)}"
        )

    row = df_plot.iloc[0]

    valor = row["qtd_ativos"]
    competencia = row["competencia"]
    pop = row["pop"]
    yoy = row["yoy"]

    valor_formatado = f"{int(round(valor, 0)):,.0f}".replace(",", ".")

    def montar_texto_variacao(nome, valor_variacao):
        if pd.isna(valor_variacao):
            return None

        seta = "▲" if valor_variacao > 0 else "▼"
        cor = "green" if valor_variacao > 0 else "red"
        texto = f"{nome} {seta} {abs(valor_variacao):.0f}%"
        return f"<span style='color:{cor}'>{texto}</span>"

    linhas_variacao = []

    if detalhe in {"completo", "pop"}:
        texto_pop = montar_texto_variacao("PoP", pop)
        if texto_pop:
            linhas_variacao.append(texto_pop)

    if detalhe in {"completo", "yoy"}:
        texto_yoy = montar_texto_variacao("YoY", yoy)
        if texto_yoy:
            linhas_variacao.append(texto_yoy)

    texto_variacoes = "<br>".join(linhas_variacao)

    fig = go.Figure()

    fig.update_layout(
        template="plotly_white",
        width=900,
        height=450,
        margin=dict(l=40, r=40, t=40, b=40),
        xaxis=dict(visible=False),
        yaxis=dict(visible=False)
    )

    # título
    fig.add_annotation(
        x=0.5,
        y=0.88,
        xref="paper",
        yref="paper",
        text=titulo,
        showarrow=False,
        font=dict(size=20, color="#1f3b64"),
        align="center"
    )

    # competência
    fig.add_annotation(
        x=0.5,
        y=0.73,
        xref="paper",
        yref="paper",
        text=f"Competência: {competencia}",
        showarrow=False,
        font=dict(size=18, color="#1f3b64"),
        align="center"
    )

    # número principal
    fig.add_annotation(
        x=0.5,
        y=0.47,
        xref="paper",
        yref="paper",
        text=valor_formatado,
        showarrow=False,
        font=dict(size=64, color="#1f3b64"),
        align="center"
    )

    # variações abaixo do número
    if detalhe != "nenhum" and texto_variacoes:
        fig.add_annotation(
            x=0.5,
            y=0.22,
            xref="paper",
            yref="paper",
            text=texto_variacoes,
            showarrow=False,
            font=dict(size=18),
            align="center"
        )

    return fig

In [211]:
df_teste_card = pd.DataFrame({
    "mes": [
        "2024-01-01","2024-02-01","2024-03-01","2024-04-01","2024-05-01","2024-06-01",
        "2024-07-01","2024-08-01","2024-09-01","2024-10-01","2024-11-01","2024-12-01",
        "2025-01-01","2025-02-01"
    ],
    "qtd_ativos": [500, 535, 550, 585, 600, 635, 668, 698, 734, 771, 806, 844, 868, 905]
})

df_teste_card["mes"] = pd.to_datetime(df_teste_card["mes"])

df_plot_teste = df_teste_card.copy()
df_plot_teste["competencia"] = df_plot_teste["mes"].dt.strftime("%Y-%m")
df_plot_teste["pop"] = df_plot_teste["qtd_ativos"].pct_change() * 100
df_plot_teste["yoy"] = df_plot_teste["qtd_ativos"].pct_change(periods=12) * 100
df_plot_teste = df_plot_teste.iloc[[-1]].copy()

fig1 = grafico_card_ativos_ultimo_mes(
    df_plot_teste,
    detalhe="completo",
    titulo="Total de ativos - teste"
)

fig1.show()

### Grafico_barras_horizontais

In [212]:
def grafico_faixa_etaria_ultimo_mes(
    df_plot,
    detalhe="completo",
    titulo="Beneficiários ativos por faixa etária - última competência"
):
    base = df_plot.copy()

    base["texto_valor"] = base["qtd_ativos"].apply(
        lambda v: f"{int(round(v, 0)):,.0f}".replace(",", ".")
    )
    base["texto_pct"] = base["percentual"].apply(
        lambda v: f"{v:.1f}%".replace(".", ",")
    )

    if detalhe == "nenhum":
        base["texto_barra"] = base["texto_valor"]
    else:
        base["texto_barra"] = base["texto_valor"] + " (" + base["texto_pct"] + ")"

    fig = px.bar(
        base,
        x="faixa_etaria",
        y="qtd_ativos",
        color="qtd_ativos",
        color_continuous_scale="Blues",
        text="texto_barra"
    )

    fig.update_traces(
        textposition="outside",
        cliponaxis=False
    )

    fig.update_layout(
        title=titulo,
        template="plotly_white",
        width=1400,
        height=800,
        margin=dict(l=50, r=50, t=80, b=80),
        xaxis_title="Faixa etária",
        yaxis_title="Quantidade de beneficiários ativos",
        coloraxis_showscale=False
    )

    fig.update_xaxes(categoryorder="array", categoryarray=base["faixa_etaria"].tolist())
    fig.update_yaxes(showgrid=False)
    fig.update_yaxes(showticklabels=False)

    return fig

In [213]:
df_teste_faixa = pd.DataFrame({
    "faixa_etaria": ["0-17", "18-24", "25-34", "35-44", "45-54", "55-64", "65-74", "75+", "Não informada"],
    "qtd_ativos": [18, 42, 120, 135, 110, 82, 48, 20, 6],
    "percentual": [3.1, 7.2, 20.6, 23.1, 18.9, 14.1, 8.2, 3.4, 1.0],
    "competencia": ["2025-02"] * 9
})

fig1 = grafico_demografia_faixa_etaria(
    df_teste_faixa,
    detalhe="completo",
    titulo="Faixa etária - teste"
)

fig1.show()

### Grafico_setor

In [214]:
def grafico_setor_sexo_ultimo_mes(
    df_plot,
    detalhe="completo",
    titulo="Distribuição de beneficiários ativos por sexo - última competência",
    largura_px=1000,
    altura_px=700,
    hole=0.62,
    percentual_minimo_fora=3.0
):
    base = df_plot.copy()

    total = base["qtd_ativos"].sum()
    competencia = base["competencia"].iloc[0] if not base.empty else ""

    cores_mapa = {
        "Feminino": "rgb(226, 115, 145)",
        "Masculino": "rgb(93, 135, 222)",
        "Não informado": "rgb(180, 180, 180)"
    }
    cores = [cores_mapa.get(cat, "rgb(200, 200, 200)") for cat in base["genero"]]

    def fmt_int(v):
        return f"{int(round(v, 0)):,}".replace(",", ".")

    def fmt_pct(v):
        return f"{v:.1f}".replace(".", ",")

    textos = []
    posicoes_texto = []
    pull = []

    for v, p in zip(base["qtd_ativos"], base["percentual"]):
        if detalhe == "nenhum":
            txt = f"{fmt_int(v)}"
        else:
            txt = f"{fmt_int(v)}<br>({fmt_pct(p)}%)"

        textos.append(txt)

        if p < percentual_minimo_fora:
            posicoes_texto.append("outside")
            pull.append(0.06)
        else:
            posicoes_texto.append("inside")
            pull.append(0)

    fig = go.Figure(
        data=[
            go.Pie(
                labels=base["genero"],
                values=base["qtd_ativos"],
                text=textos,
                textinfo="text",
                textposition=posicoes_texto,
                insidetextorientation="horizontal",
                textfont=dict(size=18, color="#2b2b2b"),
                marker=dict(colors=cores),
                hole=hole,
                pull=pull,
                sort=False,
                direction="clockwise",
                automargin=True
            )
        ]
    )

    # Centro com 3 anotações separadas para não sobrepor
    fig.add_annotation(
        x=0.5, y=0.525,
        xref="paper", yref="paper",
        text="<b>Total</b>",
        showarrow=False,
        font=dict(size=15, color="#1f3b63"),
        align="center"
    )

    fig.add_annotation(
        x=0.5, y=0.495,
        xref="paper", yref="paper",
        text=f"<b>{fmt_int(total)}</b>",
        showarrow=False,
        font=dict(size=24, color="#1f3b63"),
        align="center"
    )

    fig.add_annotation(
        x=0.5, y=0.460,
        xref="paper", yref="paper",
        text=f"Competência: {competencia}",
        showarrow=False,
        font=dict(size=13, color="#1f3b63"),
        align="center"
    )

    fig.update_layout(
        title=dict(
            text=f"<b>{titulo}</b>",
            x=0.5,
            xanchor="center",
            y=0.95,  # ↑ sobe o título
            yanchor="top",
            font=dict(size=22, color="#1f3b63")
        ),
        paper_bgcolor="white",
        plot_bgcolor="white",
        width=largura_px,
        height=altura_px,
        showlegend=True,
        legend=dict(
            font=dict(size=16),
            orientation="v",
            x=1.05,
            y=1
        ),
        margin=dict(l=40, r=220, t=120, b=40),
        uniformtext_minsize=12,
        uniformtext_mode="hide"
    )

    return fig

In [215]:
df_teste_sexo = pd.DataFrame({
    "genero": ["Feminino", "Masculino", "Não informado"],
    "qtd_ativos": [520, 380, 5],
    "percentual": [57.6, 42.0, 0.4],
    "competencia": ["2025-02", "2025-02", "2025-02"]
})

fig1 = grafico_demografia_sexo_setor(
    df_teste_sexo,
    detalhe="completo",
    titulo="Distribuição por sexo - teste"
)

fig1.show()

### Grafico_barras_verticais

In [216]:
def grafico_volumetria_mensal(
    df_plot,
    detalhe="nenhum",
    titulo="Atendimentos por competência"
):
    base = df_plot.copy()

    media_mensal = base["media_mensal"].iloc[0] if not base.empty else 0
    maior_valor = base["maior_valor"].iloc[0] if not base.empty else 0
    menor_valor = base["menor_valor"].iloc[0] if not base.empty else 0

    cor_acima_media = "rgb(20, 27, 77)"
    cor_abaixo_media = "rgb(169, 169, 169)"
    cor_menor_valor = "rgb(228, 46, 68)"
    cor_maior_valor = "rgb(65, 134, 84)"

    def determinar_cor(valor):
        if valor == maior_valor:
            return cor_maior_valor
        elif valor == menor_valor:
            return cor_menor_valor
        elif valor > media_mensal:
            return cor_acima_media
        else:
            return cor_abaixo_media

    fig = go.Figure()

    fig.add_trace(go.Bar(
        x=base["competencia"],
        y=base["volume"],
        width=0.55,
        marker_color=[determinar_cor(v) for v in base["volume"]],
        text=base["texto_valor"],
        textfont_size=14,
        textangle=90,
        textposition="inside",
        showlegend=False,
        name="Volume"
    ))

    fig.add_trace(go.Scatter(
        x=base["competencia"],
        y=base["tendencia"],
        mode="lines",
        name="Tendência",
        line=dict(color="blue", dash="dash", width=2)
    ))

    for _, row in base.iterrows():
        variacao = row["variacao_percentual"]
        if not np.isnan(variacao):
            variacao_int = int(round(abs(variacao), 0))
            cor_texto = "green" if variacao > 0 else "red"
            seta = "▲" if variacao > 0 else "▼"
            texto_variacao = f"PoP {seta}<br>{variacao_int}%"

            fig.add_annotation(
                x=row["competencia"],
                y=row["volume"],
                text=texto_variacao,
                showarrow=False,
                font=dict(size=11, color=cor_texto),
                align="center",
                yshift=55
            )

    fig.update_layout(
        title={
            "text": f"<b>{titulo}</b>",
            "y": 0.95,
            "x": 0.5,
            "xanchor": "center",
            "yanchor": "top",
            "font": {"size": 24}
        },
        xaxis_title="",
        yaxis_title="",
        barmode="stack",
        paper_bgcolor="white",
        plot_bgcolor="white",
        width=1300,
        height=800,
        bargap=0.35,
        bargroupgap=0.1,
        margin=dict(l=40, r=40, t=80, b=170),
        legend=dict(
            orientation="h",
            yanchor="bottom",
            y=1.02,
            xanchor="right",
            x=1
        )
    )

    fig.update_xaxes(
        type="category",
        tickangle=90,
        tickfont=dict(size=12),
        tickmode="array",
        tickvals=base["competencia"],
        ticktext=base["competencia"],
        showgrid=False
    )

    fig.update_yaxes(showticklabels=False)

    media_txt = f"{media_mensal:,.2f}".replace(",", "X").replace(".", ",").replace("X", ".")

    fig.add_hline(
        y=media_mensal,
        line_dash="dash",
        line_color="lightgray",
    )

    fig.add_annotation(
        x=1.01,
        xref="paper",
        y=media_mensal,
        yref="y",
        text=f"<b>Média: {media_txt}</b>",
        showarrow=False,
        font=dict(size=12, color="black"),
        align="left",
        bgcolor="rgba(255,255,255,0.8)"
    )

    return fig

In [217]:
df_teste_volumetria = pd.DataFrame({
    "mes": [
        "2024-01-01", "2024-02-01", "2024-03-01", "2024-04-01",
        "2024-05-01", "2024-06-01", "2024-07-01", "2024-08-01",
        "2024-09-01", "2024-10-01", "2024-11-01", "2024-12-01"
    ],
    "qtd_atendimentos": [
        320, 345, 380, 360,
        410, 430, 395, 450,
        470, 490, 520, 505
    ]
})

df_teste_volumetria["mes"] = pd.to_datetime(df_teste_volumetria["mes"])

df_plot_volumetria = preparar_volumetria_resumo_mensal(
    df_teste_volumetria,
    col_valor="qtd_atendimentos"
)

fig1 = grafico_volumetria_mensal(
    df_plot=df_plot_volumetria,
    titulo="Atendimentos por competência - teste manual"
)

fig1.show()

### Grafico_heatmap

In [218]:
def grafico_heatmap_dia_semana_hora(
    df_plot,
    detalhe="nenhum",
    titulo="Distribuição de atendimentos por dia da semana e hora"
):
    base = df_plot.copy()

    matriz = base["matriz"]
    barras_topo = base["barras_topo"]
    barras_lado = base["barras_lado"]
    total_atendimentos = base["total_atendimentos"]

    dias = list(matriz.columns)
    horas = list(matriz.index)

    z = matriz.values

    totais_dia = barras_topo["proporcao_total"].values
    totais_hora = barras_lado["proporcao_total"].values

    max_topo = totais_dia.max() if len(totais_dia) > 0 else 0
    max_lado = totais_hora.max() if len(totais_hora) > 0 else 0

    fig = make_subplots(
        rows=2,
        cols=2,
        row_heights=[0.28, 0.72],
        column_widths=[0.72, 0.28],
        horizontal_spacing=0.06,
        vertical_spacing=0.06,
        specs=[
            [{"type": "bar"}, {"type": "xy"}],
            [{"type": "heatmap"}, {"type": "bar"}]
        ]
    )

    # Barras topo
    fig.add_trace(
        go.Bar(
            x=dias,
            y=totais_dia,
            marker=dict(
                color=totais_dia,
                colorscale="RdBu_r",
                cmin=0,
                cmax=max_topo if max_topo > 0 else 1
            ),
            showlegend=False,
            hovertemplate="Dia: %{x}<br>Proporção: %{y:.1%}<extra></extra>"
        ),
        row=1, col=1
    )

    # Heatmap central
    fig.add_trace(
        go.Heatmap(
            z=z,
            x=dias,
            y=horas,
            colorscale="RdBu_r",
            zmin=0,
            zmax=np.max(z) if np.max(z) > 0 else 1,
            showscale=False,
            hovertemplate="Dia: %{x}<br>Hora: %{y}h<br>Proporção: %{z:.2%}<extra></extra>"
        ),
        row=2, col=1
    )

    # Barras lado direito
    fig.add_trace(
        go.Bar(
            x=totais_hora,
            y=horas,
            orientation="h",
            marker=dict(
                color=totais_hora,
                colorscale="RdBu_r",
                cmin=0,
                cmax=max_lado if max_lado > 0 else 1
            ),
            showlegend=False,
            hovertemplate="Hora: %{y}h<br>Proporção: %{x:.1%}<extra></extra>"
        ),
        row=2, col=2
    )

    # Título
    fig.update_layout(
        title=dict(
            text=f"<b>{titulo}</b><br><sup>Total de atendimentos: {int(total_atendimentos):,}</sup>".replace(",", "."),
            x=0.5,
            xanchor="center",
            y=0.98,
            yanchor="top",
            font=dict(size=22)
        ),
        template="plotly_white",
        width=1400,
        height=900,
        paper_bgcolor="white",
        plot_bgcolor="white",
        margin=dict(l=60, r=40, t=90, b=40)
    )

    # Eixos topo
    fig.update_xaxes(
        row=1, col=1,
        showgrid=False,
        tickfont=dict(size=12),
        title=""
    )
    fig.update_yaxes(
        row=1, col=1,
        showgrid=False,
        showticklabels=False,
        title=""
    )

    # Eixos heatmap
    fig.update_xaxes(
        row=2, col=1,
        showgrid=False,
        tickfont=dict(size=12),
        title=""
    )
    fig.update_yaxes(
        row=2, col=1,
        autorange="reversed",
        tickmode="array",
        tickvals=horas,
        ticktext=[f"{h}h" if h != 0 else "12am" if h == 0 else str(h) for h in horas],
        title="",
        showgrid=False
    )

    # Eixos barra lateral
    fig.update_xaxes(
        row=2, col=2,
        tickformat=".0%",
        showgrid=False,
        title=""
    )
    fig.update_yaxes(
        row=2, col=2,
        autorange="reversed",
        tickmode="array",
        tickvals=horas,
        ticktext=[f"{h}h" if h != 0 else "12am" if h == 0 else str(h) for h in horas],
        title="",
        showgrid=False
    )

    return fig

## Montagem do Filtro definido

In [219]:
def montar_filtros_sql(col_empresa):
    filtros = []

    # Empresa
    if empresa_selecionada is not None and empresa_selecionada != "Todos":
        empresa_sql = empresa_selecionada.replace("'", "''")
        filtros.append(f"{col_empresa} = '{empresa_sql}'")

    # Data mínima
    if data_inicial_relatorio is not None:
        filtros.append(f"mes >= '{data_inicial_relatorio.strftime('%Y-%m-%d')}'")

    # Data máxima
    if data_final_relatorio is not None:
        filtros.append(f"mes <= '{data_final_relatorio.strftime('%Y-%m-%d')}'")

    if filtros:
        return "WHERE " + " AND ".join(filtros)

    return ""

In [220]:
print(montar_filtros_sql("empresa_pessoa"))

In [221]:
print(montar_filtros_sql("empresa_plano_saude_paciente"))

In [222]:
print(montar_filtros_sql("empresa_plano_saude_paciente"))

## Cálculo dos Indicadores

### Número de beneficiários ativos por ano-mês (V)

In [223]:
def query_demografia_dimensoes():
    where_clause = montar_filtros_sql_demografia()

    query = f"""
    WITH

    assinaturas_base AS (
      SELECT
        na.id_pessoa,
        DATE(na.data_inicio_assinatura) AS data_inicio_assinatura,
        DATE(na.data_perda)             AS data_perda
      FROM pipedrive.negocios_assinatura na
      WHERE na.id_pessoa IS NOT NULL
        AND na.data_inicio_assinatura IS NOT NULL
    ),

    pessoas_dim AS (
      SELECT
        p.id AS id_pessoa,
        p.nome_organizacao AS empresa_pessoa,
        p.genero AS genero,
        DATE(p.data_nascimento) AS data_nascimento
      FROM crm.pessoas p
      WHERE p.id IS NOT NULL
        AND (
          p.nome IS NULL
          OR LOWER(TRIM(p.nome)) NOT LIKE '%%test%%'
        )
        AND (
          p.nome_social IS NULL
          OR LOWER(TRIM(p.nome_social)) NOT LIKE '%%test%%'
        )
    ),

    assinaturas_filtradas AS (
      SELECT a.*
      FROM assinaturas_base a
      JOIN pessoas_dim p
        ON p.id_pessoa = a.id_pessoa
    ),

    mes_ref AS (
      SELECT
        m.mes,
        CASE
          WHEN m.mes = DATE_FORMAT(CURRENT_DATE(), '%%Y-%%m-01') THEN CURRENT_DATE()
          ELSE LAST_DAY(m.mes)
        END AS ref_date
      FROM (
        SELECT DISTINCT DATE_FORMAT(data_inicio_assinatura, '%%Y-%%m-01') AS mes
        FROM assinaturas_filtradas
        UNION DISTINCT
        SELECT DISTINCT DATE_FORMAT(COALESCE(data_perda, CURRENT_DATE()), '%%Y-%%m-01') AS mes
        FROM assinaturas_filtradas
      ) m
    ),

    base_mes_pessoa AS (
      SELECT
        m.mes,
        m.ref_date,
        p.id_pessoa,
        p.empresa_pessoa,
        p.genero,
        CASE
          WHEN p.data_nascimento IS NULL THEN NULL
          ELSE FLOOR(DATEDIFF(m.ref_date, p.data_nascimento) / 365.25)
        END AS idade_anos
      FROM mes_ref m
      JOIN pessoas_dim p ON 1 = 1
    ),

    ativado_no_mes AS (
      SELECT
        b.mes,
        b.ref_date,
        b.id_pessoa,
        1 AS ativado_no_mes
      FROM base_mes_pessoa b
      JOIN assinaturas_filtradas a
        ON a.id_pessoa = b.id_pessoa
       AND a.data_inicio_assinatura >= b.mes
       AND a.data_inicio_assinatura <= b.ref_date
      GROUP BY b.mes, b.ref_date, b.id_pessoa
    ),

    inativado_no_mes AS (
      SELECT
        b.mes,
        b.ref_date,
        b.id_pessoa,
        1 AS inativado_no_mes
      FROM base_mes_pessoa b
      JOIN assinaturas_filtradas a
        ON a.id_pessoa = b.id_pessoa
       AND a.data_perda IS NOT NULL
       AND a.data_perda >= b.mes
       AND a.data_perda <= b.ref_date
      GROUP BY b.mes, b.ref_date, b.id_pessoa
    ),

    ativo_no_mes AS (
      SELECT
        b.mes,
        b.ref_date,
        b.id_pessoa,
        1 AS ativo_no_mes
      FROM base_mes_pessoa b
      JOIN assinaturas_filtradas a
        ON a.id_pessoa = b.id_pessoa
       AND b.ref_date >= a.data_inicio_assinatura
       AND b.ref_date <= COALESCE(a.data_perda, b.ref_date)
      GROUP BY b.mes, b.ref_date, b.id_pessoa
    ),

    flags_mes_pessoa AS (
      SELECT
        b.mes,
        b.ref_date,
        b.id_pessoa,
        b.empresa_pessoa,
        b.genero,
        b.idade_anos,
        COALESCE(at.ativado_no_mes, 0)   AS ativado_no_mes,
        COALESCE(iv.inativado_no_mes, 0) AS inativado_no_mes,
        COALESCE(ac.ativo_no_mes, 0)     AS ativo_no_mes
      FROM base_mes_pessoa b
      LEFT JOIN ativado_no_mes at
        ON at.mes = b.mes AND at.ref_date = b.ref_date AND at.id_pessoa = b.id_pessoa
      LEFT JOIN inativado_no_mes iv
        ON iv.mes = b.mes AND iv.ref_date = b.ref_date AND iv.id_pessoa = b.id_pessoa
      LEFT JOIN ativo_no_mes ac
        ON ac.mes = b.mes AND ac.ref_date = b.ref_date AND ac.id_pessoa = b.id_pessoa
    ),

    agregado AS (
      SELECT
        CURRENT_TIMESTAMP AS data_atualizacao,
        mes,
        ref_date,
        empresa_pessoa,
        genero,
        idade_anos,
        COUNT(DISTINCT CASE WHEN ativado_no_mes   = 1 THEN id_pessoa END) AS qtd_ativados,
        COUNT(DISTINCT CASE WHEN inativado_no_mes = 1 THEN id_pessoa END) AS qtd_inativados,
        COUNT(DISTINCT CASE WHEN ativo_no_mes     = 1 THEN id_pessoa END) AS qtd_ativos
      FROM flags_mes_pessoa
      GROUP BY mes, ref_date, empresa_pessoa, genero, idade_anos
    )

    SELECT *
    FROM agregado
    {where_clause}
    ORDER BY mes, empresa_pessoa, genero, idade_anos
    """
    return query

In [224]:
def preparar_demografia_resumo_mensal(df):
    df_plot = df.copy()

    df_plot["mes"] = pd.to_datetime(df_plot["mes"], errors="coerce")

    df_plot = (
        df_plot.groupby("mes", as_index=False)[["qtd_ativados", "qtd_inativados", "qtd_ativos"]]
        .sum()
        .sort_values("mes")
        .reset_index(drop=True)
    )

    df_plot["competencia"] = df_plot["mes"].dt.strftime("%Y-%m")

    return df_plot

In [225]:
def grafico_demografia_ativos(
    df_plot,
    detalhe="completo",
    titulo="Beneficiários ativos por competência"
):
    return grafico_linha_temporal_completo(
        df=df_plot,
        x="competencia",
        y="qtd_ativos",
        titulo=titulo,
        detalhe=detalhe
    )

In [226]:
query_teste = query_demografia_dimensoes()
df_teste_base = executar_query(query_teste, engine)
df_plot_teste = preparar_demografia_resumo_mensal(df_teste_base)
display(df_plot_teste.head())
fig1 = grafico_linha_temporal_completo(df=df_plot_teste,x="competencia",y="qtd_ativos",titulo="Teste",detalhe="nenhum")
fig1.show()

,mes,qtd_ativados,qtd_inativados,qtd_ativos,competencia
0,2020-12-01,4,0,4,2020-12
1,2021-01-01,16,0,20,2021-01
2,2021-02-01,5,3,23,2021-02
3,2021-03-01,4,4,22,2021-03
4,2021-04-01,2,5,20,2021-04


### Número de beneficiários ativados e inativados por ano-mês (V)

In [227]:
def query_demografia_dimensoes():
    where_clause = montar_filtros_sql("empresa_pessoa")

    query = f"""
    WITH

    assinaturas_base AS (
      SELECT
        na.id_pessoa,
        DATE(na.data_inicio_assinatura) AS data_inicio_assinatura,
        DATE(na.data_perda)             AS data_perda
      FROM pipedrive.negocios_assinatura na
      WHERE na.id_pessoa IS NOT NULL
        AND na.data_inicio_assinatura IS NOT NULL
    ),

    pessoas_dim AS (
      SELECT
        p.id AS id_pessoa,
        p.nome_organizacao AS empresa_pessoa,
        p.genero AS genero,
        DATE(p.data_nascimento) AS data_nascimento
      FROM crm.pessoas p
      WHERE p.id IS NOT NULL
        AND (
          p.nome IS NULL
          OR LOWER(TRIM(p.nome)) NOT LIKE '%%test%%'
        )
        AND (
          p.nome_social IS NULL
          OR LOWER(TRIM(p.nome_social)) NOT LIKE '%%test%%'
        )
    ),

    assinaturas_filtradas AS (
      SELECT a.*
      FROM assinaturas_base a
      JOIN pessoas_dim p
        ON p.id_pessoa = a.id_pessoa
    ),

    mes_ref AS (
      SELECT
        m.mes,
        CASE
          WHEN m.mes = DATE_FORMAT(CURRENT_DATE(), '%%Y-%%m-01') THEN CURRENT_DATE()
          ELSE LAST_DAY(m.mes)
        END AS ref_date
      FROM (
        SELECT DISTINCT DATE_FORMAT(data_inicio_assinatura, '%%Y-%%m-01') AS mes
        FROM assinaturas_filtradas
        UNION DISTINCT
        SELECT DISTINCT DATE_FORMAT(COALESCE(data_perda, CURRENT_DATE()), '%%Y-%%m-01') AS mes
        FROM assinaturas_filtradas
      ) m
    ),

    base_mes_pessoa AS (
      SELECT
        m.mes,
        m.ref_date,
        p.id_pessoa,
        p.empresa_pessoa,
        p.genero,
        CASE
          WHEN p.data_nascimento IS NULL THEN NULL
          ELSE FLOOR(DATEDIFF(m.ref_date, p.data_nascimento) / 365.25)
        END AS idade_anos
      FROM mes_ref m
      JOIN pessoas_dim p ON 1 = 1
    ),

    ativado_no_mes AS (
      SELECT
        b.mes,
        b.ref_date,
        b.id_pessoa,
        1 AS ativado_no_mes
      FROM base_mes_pessoa b
      JOIN assinaturas_filtradas a
        ON a.id_pessoa = b.id_pessoa
       AND a.data_inicio_assinatura >= b.mes
       AND a.data_inicio_assinatura <= b.ref_date
      GROUP BY b.mes, b.ref_date, b.id_pessoa
    ),

    inativado_no_mes AS (
      SELECT
        b.mes,
        b.ref_date,
        b.id_pessoa,
        1 AS inativado_no_mes
      FROM base_mes_pessoa b
      JOIN assinaturas_filtradas a
        ON a.id_pessoa = b.id_pessoa
       AND a.data_perda IS NOT NULL
       AND a.data_perda >= b.mes
       AND a.data_perda <= b.ref_date
      GROUP BY b.mes, b.ref_date, b.id_pessoa
    ),

    ativo_no_mes AS (
      SELECT
        b.mes,
        b.ref_date,
        b.id_pessoa,
        1 AS ativo_no_mes
      FROM base_mes_pessoa b
      JOIN assinaturas_filtradas a
        ON a.id_pessoa = b.id_pessoa
       AND b.ref_date >= a.data_inicio_assinatura
       AND b.ref_date <= COALESCE(a.data_perda, b.ref_date)
      GROUP BY b.mes, b.ref_date, b.id_pessoa
    ),

    flags_mes_pessoa AS (
      SELECT
        b.mes,
        b.ref_date,
        b.id_pessoa,
        b.empresa_pessoa,
        b.genero,
        b.idade_anos,
        COALESCE(at.ativado_no_mes, 0)   AS ativado_no_mes,
        COALESCE(iv.inativado_no_mes, 0) AS inativado_no_mes,
        COALESCE(ac.ativo_no_mes, 0)     AS ativo_no_mes
      FROM base_mes_pessoa b
      LEFT JOIN ativado_no_mes at
        ON at.mes = b.mes AND at.ref_date = b.ref_date AND at.id_pessoa = b.id_pessoa
      LEFT JOIN inativado_no_mes iv
        ON iv.mes = b.mes AND iv.ref_date = b.ref_date AND iv.id_pessoa = b.id_pessoa
      LEFT JOIN ativo_no_mes ac
        ON ac.mes = b.mes AND ac.ref_date = b.ref_date AND ac.id_pessoa = b.id_pessoa
    ),

    agregado AS (
      SELECT
        CURRENT_TIMESTAMP AS data_atualizacao,
        mes,
        ref_date,
        empresa_pessoa,
        genero,
        idade_anos,
        COUNT(DISTINCT CASE WHEN ativado_no_mes   = 1 THEN id_pessoa END) AS qtd_ativados,
        COUNT(DISTINCT CASE WHEN inativado_no_mes = 1 THEN id_pessoa END) AS qtd_inativados,
        COUNT(DISTINCT CASE WHEN ativo_no_mes     = 1 THEN id_pessoa END) AS qtd_ativos
      FROM flags_mes_pessoa
      GROUP BY mes, ref_date, empresa_pessoa, genero, idade_anos
    )

    SELECT *
    FROM agregado
    {where_clause}
    ORDER BY mes, empresa_pessoa, genero, idade_anos
    """
    return query

In [228]:
def preparar_dinamica_beneficiarios(df):
    turnover_plot = df.copy()

    turnover_plot["mes"] = pd.to_datetime(turnover_plot["mes"], errors="coerce")

    turnover_plot = (
        turnover_plot.groupby("mes", as_index=False)[["qtd_ativados", "qtd_inativados", "qtd_ativos"]]
        .sum()
        .sort_values("mes")
        .reset_index(drop=True)
    )

    turnover_plot["competencia"] = turnover_plot["mes"].dt.strftime("%Y-%m")
    turnover_plot["entradas"] = turnover_plot["qtd_ativados"]
    turnover_plot["saidas"] = turnover_plot["qtd_inativados"]
    turnover_plot["ativos"] = turnover_plot["qtd_ativos"]

    return turnover_plot

In [229]:
def grafico_dinamica_beneficiarios(
    turnover_plot,
    detalhe="completo",
    titulo="Dinâmica mensal de beneficiários – Entradas e Saídas"
):
    return grafico_dinamica_entrada_saida(
        turnover_plot,
        detalhe=detalhe,
        titulo=titulo
    )

In [230]:
query_teste = query_demografia_dimensoes()
df_teste_base = executar_query(query_teste, engine)
df_plot_teste = preparar_dinamica_beneficiarios(df_teste_base)
display(df_plot_teste.head())
fig1 = grafico_dinamica_entrada_saida(df_plot_teste,detalhe="completo",titulo="Teste")
fig1.show()

,mes,qtd_ativados,qtd_inativados,qtd_ativos,competencia,entradas,saidas,ativos
0,2020-12-01,4,0,4,2020-12,4,0,4
1,2021-01-01,16,0,20,2021-01,16,0,20
2,2021-02-01,5,3,23,2021-02,5,3,23
3,2021-03-01,4,4,22,2021-03,4,4,22
4,2021-04-01,2,5,20,2021-04,2,5,20


### Número de Beneficiários total de um ano-mês específico (padrão último) (V)

In [231]:
def query_demografia_dimensoes():
    where_clause = montar_filtros_sql("empresa_pessoa")

    query = f"""
    WITH

    assinaturas_base AS (
      SELECT
        na.id_pessoa,
        DATE(na.data_inicio_assinatura) AS data_inicio_assinatura,
        DATE(na.data_perda)             AS data_perda
      FROM pipedrive.negocios_assinatura na
      WHERE na.id_pessoa IS NOT NULL
        AND na.data_inicio_assinatura IS NOT NULL
    ),

    pessoas_dim AS (
      SELECT
        p.id AS id_pessoa,
        p.nome_organizacao AS empresa_pessoa,
        p.genero AS genero,
        DATE(p.data_nascimento) AS data_nascimento
      FROM crm.pessoas p
      WHERE p.id IS NOT NULL
        AND (
          p.nome IS NULL
          OR LOWER(TRIM(p.nome)) NOT LIKE '%%test%%'
        )
        AND (
          p.nome_social IS NULL
          OR LOWER(TRIM(p.nome_social)) NOT LIKE '%%test%%'
        )
    ),

    assinaturas_filtradas AS (
      SELECT a.*
      FROM assinaturas_base a
      JOIN pessoas_dim p
        ON p.id_pessoa = a.id_pessoa
    ),

    mes_ref AS (
      SELECT
        m.mes,
        CASE
          WHEN m.mes = DATE_FORMAT(CURRENT_DATE(), '%%Y-%%m-01') THEN CURRENT_DATE()
          ELSE LAST_DAY(m.mes)
        END AS ref_date
      FROM (
        SELECT DISTINCT DATE_FORMAT(data_inicio_assinatura, '%%Y-%%m-01') AS mes
        FROM assinaturas_filtradas
        UNION DISTINCT
        SELECT DISTINCT DATE_FORMAT(COALESCE(data_perda, CURRENT_DATE()), '%%Y-%%m-01') AS mes
        FROM assinaturas_filtradas
      ) m
    ),

    base_mes_pessoa AS (
      SELECT
        m.mes,
        m.ref_date,
        p.id_pessoa,
        p.empresa_pessoa,
        p.genero,
        CASE
          WHEN p.data_nascimento IS NULL THEN NULL
          ELSE FLOOR(DATEDIFF(m.ref_date, p.data_nascimento) / 365.25)
        END AS idade_anos
      FROM mes_ref m
      JOIN pessoas_dim p ON 1 = 1
    ),

    ativado_no_mes AS (
      SELECT
        b.mes,
        b.ref_date,
        b.id_pessoa,
        1 AS ativado_no_mes
      FROM base_mes_pessoa b
      JOIN assinaturas_filtradas a
        ON a.id_pessoa = b.id_pessoa
       AND a.data_inicio_assinatura >= b.mes
       AND a.data_inicio_assinatura <= b.ref_date
      GROUP BY b.mes, b.ref_date, b.id_pessoa
    ),

    inativado_no_mes AS (
      SELECT
        b.mes,
        b.ref_date,
        b.id_pessoa,
        1 AS inativado_no_mes
      FROM base_mes_pessoa b
      JOIN assinaturas_filtradas a
        ON a.id_pessoa = b.id_pessoa
       AND a.data_perda IS NOT NULL
       AND a.data_perda >= b.mes
       AND a.data_perda <= b.ref_date
      GROUP BY b.mes, b.ref_date, b.id_pessoa
    ),

    ativo_no_mes AS (
      SELECT
        b.mes,
        b.ref_date,
        b.id_pessoa,
        1 AS ativo_no_mes
      FROM base_mes_pessoa b
      JOIN assinaturas_filtradas a
        ON a.id_pessoa = b.id_pessoa
       AND b.ref_date >= a.data_inicio_assinatura
       AND b.ref_date <= COALESCE(a.data_perda, b.ref_date)
      GROUP BY b.mes, b.ref_date, b.id_pessoa
    ),

    flags_mes_pessoa AS (
      SELECT
        b.mes,
        b.ref_date,
        b.id_pessoa,
        b.empresa_pessoa,
        b.genero,
        b.idade_anos,
        COALESCE(at.ativado_no_mes, 0)   AS ativado_no_mes,
        COALESCE(iv.inativado_no_mes, 0) AS inativado_no_mes,
        COALESCE(ac.ativo_no_mes, 0)     AS ativo_no_mes
      FROM base_mes_pessoa b
      LEFT JOIN ativado_no_mes at
        ON at.mes = b.mes AND at.ref_date = b.ref_date AND at.id_pessoa = b.id_pessoa
      LEFT JOIN inativado_no_mes iv
        ON iv.mes = b.mes AND iv.ref_date = b.ref_date AND iv.id_pessoa = b.id_pessoa
      LEFT JOIN ativo_no_mes ac
        ON ac.mes = b.mes AND ac.ref_date = b.ref_date AND ac.id_pessoa = b.id_pessoa
    ),

    agregado AS (
      SELECT
        CURRENT_TIMESTAMP AS data_atualizacao,
        mes,
        ref_date,
        empresa_pessoa,
        genero,
        idade_anos,
        COUNT(DISTINCT CASE WHEN ativado_no_mes   = 1 THEN id_pessoa END) AS qtd_ativados,
        COUNT(DISTINCT CASE WHEN inativado_no_mes = 1 THEN id_pessoa END) AS qtd_inativados,
        COUNT(DISTINCT CASE WHEN ativo_no_mes     = 1 THEN id_pessoa END) AS qtd_ativos
      FROM flags_mes_pessoa
      GROUP BY mes, ref_date, empresa_pessoa, genero, idade_anos
    )

    SELECT *
    FROM agregado
    {where_clause}
    ORDER BY mes, empresa_pessoa, genero, idade_anos
    """
    return query

In [232]:
def preparar_card_ativos_ultimo_mes(df):
    base = df.copy()

    base["mes"] = pd.to_datetime(base["mes"], errors="coerce")

    resumo = (
        base.groupby("mes", as_index=False)[["qtd_ativos"]]
        .sum()
        .sort_values("mes")
        .reset_index(drop=True)
    )

    resumo["competencia"] = resumo["mes"].dt.strftime("%Y-%m")
    resumo["pop"] = resumo["qtd_ativos"].pct_change() * 100
    resumo["yoy"] = resumo["qtd_ativos"].pct_change(periods=12) * 100

    ultimo = resumo.iloc[[-1]].copy()

    return ultimo

In [233]:
def grafico_card_total_ativos(
    df_plot,
    detalhe="completo",
    titulo="Total de beneficiários ativos - última competência"
):
    return grafico_card_ativos_ultimo_mes(
        df_plot=df_plot,
        detalhe=detalhe,
        titulo=titulo
    )

In [234]:
query_teste = query_demografia_dimensoes()
df_teste = executar_query(query_teste, engine)
df_plot_teste = preparar_card_ativos_ultimo_mes(df_teste)
display(df_plot_teste)
fig1 = grafico_card_total_ativos(df_plot_teste,detalhe="completo",titulo="Total de beneficiários ativos - última competência")
fig1.show()

,mes,qtd_ativos,competencia,pop,yoy
61,2026-03-01,535,2026-03,0.0,-6.794425


### Número de Beneficiários de um ano-mês específico (padrão último) por sexo e idade (V) 

In [235]:
def query_demografia_dimensoes():
    where_clause = montar_filtros_sql("empresa_pessoa")

    query = f"""
    WITH

    assinaturas_base AS (
      SELECT
        na.id_pessoa,
        DATE(na.data_inicio_assinatura) AS data_inicio_assinatura,
        DATE(na.data_perda)             AS data_perda
      FROM pipedrive.negocios_assinatura na
      WHERE na.id_pessoa IS NOT NULL
        AND na.data_inicio_assinatura IS NOT NULL
    ),

    pessoas_dim AS (
      SELECT
        p.id AS id_pessoa,
        p.nome_organizacao AS empresa_pessoa,
        p.genero AS genero,
        DATE(p.data_nascimento) AS data_nascimento
      FROM crm.pessoas p
      WHERE p.id IS NOT NULL
        AND (
          p.nome IS NULL
          OR LOWER(TRIM(p.nome)) NOT LIKE '%%test%%'
        )
        AND (
          p.nome_social IS NULL
          OR LOWER(TRIM(p.nome_social)) NOT LIKE '%%test%%'
        )
    ),

    assinaturas_filtradas AS (
      SELECT a.*
      FROM assinaturas_base a
      JOIN pessoas_dim p
        ON p.id_pessoa = a.id_pessoa
    ),

    mes_ref AS (
      SELECT
        m.mes,
        CASE
          WHEN m.mes = DATE_FORMAT(CURRENT_DATE(), '%%Y-%%m-01') THEN CURRENT_DATE()
          ELSE LAST_DAY(m.mes)
        END AS ref_date
      FROM (
        SELECT DISTINCT DATE_FORMAT(data_inicio_assinatura, '%%Y-%%m-01') AS mes
        FROM assinaturas_filtradas
        UNION DISTINCT
        SELECT DISTINCT DATE_FORMAT(COALESCE(data_perda, CURRENT_DATE()), '%%Y-%%m-01') AS mes
        FROM assinaturas_filtradas
      ) m
    ),

    base_mes_pessoa AS (
      SELECT
        m.mes,
        m.ref_date,
        p.id_pessoa,
        p.empresa_pessoa,
        p.genero,
        CASE
          WHEN p.data_nascimento IS NULL THEN NULL
          ELSE FLOOR(DATEDIFF(m.ref_date, p.data_nascimento) / 365.25)
        END AS idade_anos
      FROM mes_ref m
      JOIN pessoas_dim p ON 1 = 1
    ),

    ativado_no_mes AS (
      SELECT
        b.mes,
        b.ref_date,
        b.id_pessoa,
        1 AS ativado_no_mes
      FROM base_mes_pessoa b
      JOIN assinaturas_filtradas a
        ON a.id_pessoa = b.id_pessoa
       AND a.data_inicio_assinatura >= b.mes
       AND a.data_inicio_assinatura <= b.ref_date
      GROUP BY b.mes, b.ref_date, b.id_pessoa
    ),

    inativado_no_mes AS (
      SELECT
        b.mes,
        b.ref_date,
        b.id_pessoa,
        1 AS inativado_no_mes
      FROM base_mes_pessoa b
      JOIN assinaturas_filtradas a
        ON a.id_pessoa = b.id_pessoa
       AND a.data_perda IS NOT NULL
       AND a.data_perda >= b.mes
       AND a.data_perda <= b.ref_date
      GROUP BY b.mes, b.ref_date, b.id_pessoa
    ),

    ativo_no_mes AS (
      SELECT
        b.mes,
        b.ref_date,
        b.id_pessoa,
        1 AS ativo_no_mes
      FROM base_mes_pessoa b
      JOIN assinaturas_filtradas a
        ON a.id_pessoa = b.id_pessoa
       AND b.ref_date >= a.data_inicio_assinatura
       AND b.ref_date <= COALESCE(a.data_perda, b.ref_date)
      GROUP BY b.mes, b.ref_date, b.id_pessoa
    ),

    flags_mes_pessoa AS (
      SELECT
        b.mes,
        b.ref_date,
        b.id_pessoa,
        b.empresa_pessoa,
        b.genero,
        b.idade_anos,
        COALESCE(at.ativado_no_mes, 0)   AS ativado_no_mes,
        COALESCE(iv.inativado_no_mes, 0) AS inativado_no_mes,
        COALESCE(ac.ativo_no_mes, 0)     AS ativo_no_mes
      FROM base_mes_pessoa b
      LEFT JOIN ativado_no_mes at
        ON at.mes = b.mes AND at.ref_date = b.ref_date AND at.id_pessoa = b.id_pessoa
      LEFT JOIN inativado_no_mes iv
        ON iv.mes = b.mes AND iv.ref_date = b.ref_date AND iv.id_pessoa = b.id_pessoa
      LEFT JOIN ativo_no_mes ac
        ON ac.mes = b.mes AND ac.ref_date = b.ref_date AND ac.id_pessoa = b.id_pessoa
    ),

    agregado AS (
      SELECT
        CURRENT_TIMESTAMP AS data_atualizacao,
        mes,
        ref_date,
        empresa_pessoa,
        genero,
        idade_anos,
        COUNT(DISTINCT CASE WHEN ativado_no_mes   = 1 THEN id_pessoa END) AS qtd_ativados,
        COUNT(DISTINCT CASE WHEN inativado_no_mes = 1 THEN id_pessoa END) AS qtd_inativados,
        COUNT(DISTINCT CASE WHEN ativo_no_mes     = 1 THEN id_pessoa END) AS qtd_ativos
      FROM flags_mes_pessoa
      GROUP BY mes, ref_date, empresa_pessoa, genero, idade_anos
    )

    SELECT *
    FROM agregado
    {where_clause}
    ORDER BY mes, empresa_pessoa, genero, idade_anos
    """
    return query

In [236]:
def preparar_piramide_etaria_ultimo_mes(df):
    base = df.copy()

    base["mes"] = pd.to_datetime(base["mes"], errors="coerce")
    base["idade_anos"] = pd.to_numeric(base["idade_anos"], errors="coerce")

    # manter só registros com ativos > 0
    base = base[base["qtd_ativos"].fillna(0) > 0].copy()

    # pegar último mês disponível
    ultimo_mes = base["mes"].max()
    base = base[base["mes"] == ultimo_mes].copy()

    # normalizar gênero
    base["genero"] = (
        base["genero"]
        .fillna("Não informado")
        .astype(str)
        .str.strip()
        .str.lower()
    )

    def normalizar_genero(g):
        if g in ["masculino", "masc", "m"]:
            return "Masculino"
        elif g in ["feminino", "fem", "f"]:
            return "Feminino"
        else:
            return "Não informado"

    base["genero"] = base["genero"].apply(normalizar_genero)

    # criar faixas etárias
    bins = [0, 18, 25, 35, 45, 55, 65, 75, 200]
    labels = ["0-17", "18-24", "25-34", "35-44", "45-54", "55-64", "65-74", "75+"]

    base["faixa_etaria"] = pd.cut(
        base["idade_anos"],
        bins=bins,
        labels=labels,
        right=False,
        include_lowest=True
    )

    # idade ausente
    base["faixa_etaria"] = base["faixa_etaria"].astype(object)
    base.loc[base["idade_anos"].isna(), "faixa_etaria"] = "Não informada"

    ordem_faixas = ["0-17", "18-24", "25-34", "35-44", "45-54", "55-64", "65-74", "75+", "Não informada"]

    resumo = (
        base.groupby(["faixa_etaria", "genero"], as_index=False)["qtd_ativos"]
        .sum()
    )

    # garantir combinações
    generos = ["Masculino", "Feminino", "Não informado"]
    grade = pd.MultiIndex.from_product(
        [ordem_faixas, generos],
        names=["faixa_etaria", "genero"]
    ).to_frame(index=False)

    resumo = grade.merge(
        resumo,
        on=["faixa_etaria", "genero"],
        how="left"
    )

    resumo["qtd_ativos"] = resumo["qtd_ativos"].fillna(0)

    resumo["faixa_etaria"] = pd.Categorical(
        resumo["faixa_etaria"],
        categories=ordem_faixas,
        ordered=True
    )

    resumo = resumo.sort_values("faixa_etaria").reset_index(drop=True)
    resumo["mes_referencia"] = ultimo_mes

    return resumo

In [237]:
def grafico_demografia_piramide_etaria(
    df_plot,
    detalhe="completo",
    titulo="Pirâmide etária por sexo - último mês"
):
    return grafico_piramide_etaria(
        df_plot=df_plot,
        titulo=titulo,
        incluir_nao_informado=True
    )

In [238]:
query_teste = query_demografia_dimensoes()
df_teste_base = executar_query(query_teste, engine)
df_plot_teste = preparar_piramide_etaria_ultimo_mes(df_teste_base)
display(df_plot_teste .head())
fig1 = grafico_demografia_piramide_etaria(df_plot_teste,titulo="Teste")
fig1.show()

,faixa_etaria,genero,qtd_ativos,mes_referencia
0,0-17,Masculino,2.0,2026-03-01
1,0-17,Feminino,5.0,2026-03-01
2,0-17,Não informado,0.0,2026-03-01
3,18-24,Masculino,31.0,2026-03-01
4,18-24,Feminino,25.0,2026-03-01


###  Número de Beneficiários de um ano-mês específico (padrão último) por faixa etária (V)

In [239]:
def query_demografia_dimensoes():
    where_clause = montar_filtros_sql("empresa_pessoa")

    query = f"""
    WITH

    assinaturas_base AS (
      SELECT
        na.id_pessoa,
        DATE(na.data_inicio_assinatura) AS data_inicio_assinatura,
        DATE(na.data_perda)             AS data_perda
      FROM pipedrive.negocios_assinatura na
      WHERE na.id_pessoa IS NOT NULL
        AND na.data_inicio_assinatura IS NOT NULL
    ),

    pessoas_dim AS (
      SELECT
        p.id AS id_pessoa,
        p.nome_organizacao AS empresa_pessoa,
        p.genero AS genero,
        DATE(p.data_nascimento) AS data_nascimento
      FROM crm.pessoas p
      WHERE p.id IS NOT NULL
        AND (
          p.nome IS NULL
          OR LOWER(TRIM(p.nome)) NOT LIKE '%%test%%'
        )
        AND (
          p.nome_social IS NULL
          OR LOWER(TRIM(p.nome_social)) NOT LIKE '%%test%%'
        )
    ),

    assinaturas_filtradas AS (
      SELECT a.*
      FROM assinaturas_base a
      JOIN pessoas_dim p
        ON p.id_pessoa = a.id_pessoa
    ),

    mes_ref AS (
      SELECT
        m.mes,
        CASE
          WHEN m.mes = DATE_FORMAT(CURRENT_DATE(), '%%Y-%%m-01') THEN CURRENT_DATE()
          ELSE LAST_DAY(m.mes)
        END AS ref_date
      FROM (
        SELECT DISTINCT DATE_FORMAT(data_inicio_assinatura, '%%Y-%%m-01') AS mes
        FROM assinaturas_filtradas
        UNION DISTINCT
        SELECT DISTINCT DATE_FORMAT(COALESCE(data_perda, CURRENT_DATE()), '%%Y-%%m-01') AS mes
        FROM assinaturas_filtradas
      ) m
    ),

    base_mes_pessoa AS (
      SELECT
        m.mes,
        m.ref_date,
        p.id_pessoa,
        p.empresa_pessoa,
        p.genero,
        CASE
          WHEN p.data_nascimento IS NULL THEN NULL
          ELSE FLOOR(DATEDIFF(m.ref_date, p.data_nascimento) / 365.25)
        END AS idade_anos
      FROM mes_ref m
      JOIN pessoas_dim p ON 1 = 1
    ),

    ativado_no_mes AS (
      SELECT
        b.mes,
        b.ref_date,
        b.id_pessoa,
        1 AS ativado_no_mes
      FROM base_mes_pessoa b
      JOIN assinaturas_filtradas a
        ON a.id_pessoa = b.id_pessoa
       AND a.data_inicio_assinatura >= b.mes
       AND a.data_inicio_assinatura <= b.ref_date
      GROUP BY b.mes, b.ref_date, b.id_pessoa
    ),

    inativado_no_mes AS (
      SELECT
        b.mes,
        b.ref_date,
        b.id_pessoa,
        1 AS inativado_no_mes
      FROM base_mes_pessoa b
      JOIN assinaturas_filtradas a
        ON a.id_pessoa = b.id_pessoa
       AND a.data_perda IS NOT NULL
       AND a.data_perda >= b.mes
       AND a.data_perda <= b.ref_date
      GROUP BY b.mes, b.ref_date, b.id_pessoa
    ),

    ativo_no_mes AS (
      SELECT
        b.mes,
        b.ref_date,
        b.id_pessoa,
        1 AS ativo_no_mes
      FROM base_mes_pessoa b
      JOIN assinaturas_filtradas a
        ON a.id_pessoa = b.id_pessoa
       AND b.ref_date >= a.data_inicio_assinatura
       AND b.ref_date <= COALESCE(a.data_perda, b.ref_date)
      GROUP BY b.mes, b.ref_date, b.id_pessoa
    ),

    flags_mes_pessoa AS (
      SELECT
        b.mes,
        b.ref_date,
        b.id_pessoa,
        b.empresa_pessoa,
        b.genero,
        b.idade_anos,
        COALESCE(at.ativado_no_mes, 0)   AS ativado_no_mes,
        COALESCE(iv.inativado_no_mes, 0) AS inativado_no_mes,
        COALESCE(ac.ativo_no_mes, 0)     AS ativo_no_mes
      FROM base_mes_pessoa b
      LEFT JOIN ativado_no_mes at
        ON at.mes = b.mes AND at.ref_date = b.ref_date AND at.id_pessoa = b.id_pessoa
      LEFT JOIN inativado_no_mes iv
        ON iv.mes = b.mes AND iv.ref_date = b.ref_date AND iv.id_pessoa = b.id_pessoa
      LEFT JOIN ativo_no_mes ac
        ON ac.mes = b.mes AND ac.ref_date = b.ref_date AND ac.id_pessoa = b.id_pessoa
    ),

    agregado AS (
      SELECT
        CURRENT_TIMESTAMP AS data_atualizacao,
        mes,
        ref_date,
        empresa_pessoa,
        genero,
        idade_anos,
        COUNT(DISTINCT CASE WHEN ativado_no_mes   = 1 THEN id_pessoa END) AS qtd_ativados,
        COUNT(DISTINCT CASE WHEN inativado_no_mes = 1 THEN id_pessoa END) AS qtd_inativados,
        COUNT(DISTINCT CASE WHEN ativo_no_mes     = 1 THEN id_pessoa END) AS qtd_ativos
      FROM flags_mes_pessoa
      GROUP BY mes, ref_date, empresa_pessoa, genero, idade_anos
    )

    SELECT *
    FROM agregado
    {where_clause}
    ORDER BY mes, empresa_pessoa, genero, idade_anos
    """
    return query

In [240]:
def preparar_faixa_etaria_ultimo_mes(df):
    base = df.copy()

    base["mes"] = pd.to_datetime(base["mes"], errors="coerce")
    base["idade_anos"] = pd.to_numeric(base["idade_anos"], errors="coerce")
    base["qtd_ativos"] = pd.to_numeric(base["qtd_ativos"], errors="coerce").fillna(0)

    # pegar último mês disponível
    ultimo_mes = base["mes"].max()
    base = base[base["mes"] == ultimo_mes].copy()

    # criar faixas etárias
    bins = [0, 18, 25, 35, 45, 55, 65, 75, 200]
    labels = ["0-17", "18-24", "25-34", "35-44", "45-54", "55-64", "65-74", "75+"]

    base["faixa_etaria"] = pd.cut(
        base["idade_anos"],
        bins=bins,
        labels=labels,
        right=False,
        include_lowest=True
    )

    base["faixa_etaria"] = base["faixa_etaria"].astype(object)
    base.loc[base["idade_anos"].isna(), "faixa_etaria"] = "Não informada"

    ordem_faixas = ["0-17", "18-24", "25-34", "35-44", "45-54", "55-64", "65-74", "75+", "Não informada"]

    resumo = (
        base.groupby("faixa_etaria", as_index=False)["qtd_ativos"]
        .sum()
    )

    # garantir todas as faixas
    resumo = pd.DataFrame({"faixa_etaria": ordem_faixas}).merge(
        resumo,
        on="faixa_etaria",
        how="left"
    )

    resumo["qtd_ativos"] = resumo["qtd_ativos"].fillna(0)

    total = resumo["qtd_ativos"].sum()
    if total > 0:
        resumo["percentual"] = resumo["qtd_ativos"] / total * 100
    else:
        resumo["percentual"] = 0

    resumo["mes_referencia"] = ultimo_mes
    resumo["competencia"] = pd.to_datetime(ultimo_mes).strftime("%Y-%m")

    return resumo

In [241]:
def grafico_demografia_faixa_etaria(
    df_plot,
    detalhe="completo",
    titulo="Beneficiários ativos por faixa etária - última competência"
):
    return grafico_faixa_etaria_ultimo_mes(
        df_plot=df_plot,
        detalhe=detalhe,
        titulo=titulo
    )

In [242]:
query_teste = query_demografia_dimensoes()
df_teste = executar_query(query_teste, engine)
df_plot_teste = preparar_faixa_etaria_ultimo_mes(df_teste)
display(df_plot_teste)
fig1 = grafico_demografia_faixa_etaria(df_plot_teste,detalhe="completo",titulo="Beneficiários ativos por faixa etária - última competência")
fig1.show()

,faixa_etaria,qtd_ativos,percentual,mes_referencia,competencia
0,0-17,7,1.308411,2026-03-01,2026-03
1,18-24,61,11.401869,2026-03-01,2026-03
2,25-34,249,46.542056,2026-03-01,2026-03
3,35-44,129,24.112150,2026-03-01,2026-03
4,45-54,59,11.028037,2026-03-01,2026-03
5,55-64,23,4.299065,2026-03-01,2026-03
6,65-74,1,0.186916,2026-03-01,2026-03
7,75+,3,0.560748,2026-03-01,2026-03
8,Não informada,3,0.560748,2026-03-01,2026-03


In [243]:
query_teste = query_demografia_dimensoes()
df_teste_base = executar_query(query_teste, engine)
df_plot_teste = preparar_piramide_etaria_ultimo_mes(df_teste_base)
display(df_plot_teste .head())
fig1 = grafico_demografia_piramide_etaria(df_plot_teste,titulo="Teste")
fig1.show()

,faixa_etaria,genero,qtd_ativos,mes_referencia
0,0-17,Masculino,2.0,2026-03-01
1,0-17,Feminino,5.0,2026-03-01
2,0-17,Não informado,0.0,2026-03-01
3,18-24,Masculino,31.0,2026-03-01
4,18-24,Feminino,25.0,2026-03-01


### Número de Beneficiários de um ano-mês específico (padrão último) por sexo (V)

In [244]:
def query_demografia_dimensoes():
    where_clause = montar_filtros_sql("empresa_pessoa")

    query = f"""
    WITH

    assinaturas_base AS (
      SELECT
        na.id_pessoa,
        DATE(na.data_inicio_assinatura) AS data_inicio_assinatura,
        DATE(na.data_perda)             AS data_perda
      FROM pipedrive.negocios_assinatura na
      WHERE na.id_pessoa IS NOT NULL
        AND na.data_inicio_assinatura IS NOT NULL
    ),

    pessoas_dim AS (
      SELECT
        p.id AS id_pessoa,
        p.nome_organizacao AS empresa_pessoa,
        p.genero AS genero,
        DATE(p.data_nascimento) AS data_nascimento
      FROM crm.pessoas p
      WHERE p.id IS NOT NULL
        AND (
          p.nome IS NULL
          OR LOWER(TRIM(p.nome)) NOT LIKE '%%test%%'
        )
        AND (
          p.nome_social IS NULL
          OR LOWER(TRIM(p.nome_social)) NOT LIKE '%%test%%'
        )
    ),

    assinaturas_filtradas AS (
      SELECT a.*
      FROM assinaturas_base a
      JOIN pessoas_dim p
        ON p.id_pessoa = a.id_pessoa
    ),

    mes_ref AS (
      SELECT
        m.mes,
        CASE
          WHEN m.mes = DATE_FORMAT(CURRENT_DATE(), '%%Y-%%m-01') THEN CURRENT_DATE()
          ELSE LAST_DAY(m.mes)
        END AS ref_date
      FROM (
        SELECT DISTINCT DATE_FORMAT(data_inicio_assinatura, '%%Y-%%m-01') AS mes
        FROM assinaturas_filtradas
        UNION DISTINCT
        SELECT DISTINCT DATE_FORMAT(COALESCE(data_perda, CURRENT_DATE()), '%%Y-%%m-01') AS mes
        FROM assinaturas_filtradas
      ) m
    ),

    base_mes_pessoa AS (
      SELECT
        m.mes,
        m.ref_date,
        p.id_pessoa,
        p.empresa_pessoa,
        p.genero,
        CASE
          WHEN p.data_nascimento IS NULL THEN NULL
          ELSE FLOOR(DATEDIFF(m.ref_date, p.data_nascimento) / 365.25)
        END AS idade_anos
      FROM mes_ref m
      JOIN pessoas_dim p ON 1 = 1
    ),

    ativado_no_mes AS (
      SELECT
        b.mes,
        b.ref_date,
        b.id_pessoa,
        1 AS ativado_no_mes
      FROM base_mes_pessoa b
      JOIN assinaturas_filtradas a
        ON a.id_pessoa = b.id_pessoa
       AND a.data_inicio_assinatura >= b.mes
       AND a.data_inicio_assinatura <= b.ref_date
      GROUP BY b.mes, b.ref_date, b.id_pessoa
    ),

    inativado_no_mes AS (
      SELECT
        b.mes,
        b.ref_date,
        b.id_pessoa,
        1 AS inativado_no_mes
      FROM base_mes_pessoa b
      JOIN assinaturas_filtradas a
        ON a.id_pessoa = b.id_pessoa
       AND a.data_perda IS NOT NULL
       AND a.data_perda >= b.mes
       AND a.data_perda <= b.ref_date
      GROUP BY b.mes, b.ref_date, b.id_pessoa
    ),

    ativo_no_mes AS (
      SELECT
        b.mes,
        b.ref_date,
        b.id_pessoa,
        1 AS ativo_no_mes
      FROM base_mes_pessoa b
      JOIN assinaturas_filtradas a
        ON a.id_pessoa = b.id_pessoa
       AND b.ref_date >= a.data_inicio_assinatura
       AND b.ref_date <= COALESCE(a.data_perda, b.ref_date)
      GROUP BY b.mes, b.ref_date, b.id_pessoa
    ),

    flags_mes_pessoa AS (
      SELECT
        b.mes,
        b.ref_date,
        b.id_pessoa,
        b.empresa_pessoa,
        b.genero,
        b.idade_anos,
        COALESCE(at.ativado_no_mes, 0)   AS ativado_no_mes,
        COALESCE(iv.inativado_no_mes, 0) AS inativado_no_mes,
        COALESCE(ac.ativo_no_mes, 0)     AS ativo_no_mes
      FROM base_mes_pessoa b
      LEFT JOIN ativado_no_mes at
        ON at.mes = b.mes AND at.ref_date = b.ref_date AND at.id_pessoa = b.id_pessoa
      LEFT JOIN inativado_no_mes iv
        ON iv.mes = b.mes AND iv.ref_date = b.ref_date AND iv.id_pessoa = b.id_pessoa
      LEFT JOIN ativo_no_mes ac
        ON ac.mes = b.mes AND ac.ref_date = b.ref_date AND ac.id_pessoa = b.id_pessoa
    ),

    agregado AS (
      SELECT
        CURRENT_TIMESTAMP AS data_atualizacao,
        mes,
        ref_date,
        empresa_pessoa,
        genero,
        idade_anos,
        COUNT(DISTINCT CASE WHEN ativado_no_mes   = 1 THEN id_pessoa END) AS qtd_ativados,
        COUNT(DISTINCT CASE WHEN inativado_no_mes = 1 THEN id_pessoa END) AS qtd_inativados,
        COUNT(DISTINCT CASE WHEN ativo_no_mes     = 1 THEN id_pessoa END) AS qtd_ativos
      FROM flags_mes_pessoa
      GROUP BY mes, ref_date, empresa_pessoa, genero, idade_anos
    )

    SELECT *
    FROM agregado
    {where_clause}
    ORDER BY mes, empresa_pessoa, genero, idade_anos
    """
    return query

In [245]:
def preparar_sexo_ultimo_mes(df):
    base = df.copy()

    base["mes"] = pd.to_datetime(base["mes"], errors="coerce")
    base["qtd_ativos"] = pd.to_numeric(base["qtd_ativos"], errors="coerce").fillna(0)

    # último mês disponível
    ultimo_mes = base["mes"].max()
    base = base[base["mes"] == ultimo_mes].copy()

    # normalizar sexo/gênero
    base["genero"] = (
        base["genero"]
        .fillna("Não informado")
        .astype(str)
        .str.strip()
        .str.lower()
    )

    def normalizar_genero(g):
        if g in ["masculino", "masc", "m"]:
            return "Masculino"
        elif g in ["feminino", "fem", "f"]:
            return "Feminino"
        else:
            return "Não informado"

    base["genero"] = base["genero"].apply(normalizar_genero)

    resumo = (
        base.groupby("genero", as_index=False)["qtd_ativos"]
        .sum()
    )

    ordem_genero = ["Feminino", "Masculino", "Não informado"]
    resumo["genero"] = pd.Categorical(
        resumo["genero"],
        categories=ordem_genero,
        ordered=True
    )
    resumo = resumo.sort_values("genero").reset_index(drop=True)

    total = resumo["qtd_ativos"].sum()
    resumo["percentual"] = 0 if total == 0 else resumo["qtd_ativos"] / total * 100
    resumo["competencia"] = pd.to_datetime(ultimo_mes).strftime("%Y-%m")

    return resumo

In [246]:
def grafico_demografia_sexo_setor(
    df_plot,
    detalhe="completo",
    titulo="Distribuição de beneficiários ativos por sexo - última competência"
):
    return grafico_setor_sexo_ultimo_mes(
        df_plot=df_plot,
        detalhe=detalhe,
        titulo=titulo
    )

In [247]:
query_teste = query_demografia_dimensoes()
df_teste = executar_query(query_teste, engine)
df_plot_teste = preparar_sexo_ultimo_mes(df_teste)
display(df_teste.head())
display(df_plot_teste)
fig1 = grafico_demografia_sexo_setor(df_plot_teste,detalhe="completo",titulo="Distribuição de beneficiários ativos por sexo - última competência")
fig1.show()

,data_atualizacao,mes,ref_date,empresa_pessoa,genero,idade_anos,qtd_ativados,qtd_inativados,qtd_ativos
0,2026-03-20 18:17:00,2020-12-01,2020-12-31,None,None,NaN,0,0,0
1,2026-03-20 18:17:00,2020-12-01,2020-12-31,None,None,16.0,0,0,0
2,2026-03-20 18:17:00,2020-12-01,2020-12-31,None,None,20.0,0,0,0
3,2026-03-20 18:17:00,2020-12-01,2020-12-31,None,None,22.0,0,0,0
4,2026-03-20 18:17:00,2020-12-01,2020-12-31,None,None,23.0,0,0,0


,genero,qtd_ativos,percentual,competencia
0,Feminino,241,45.046729,2026-03
1,Masculino,246,45.981308,2026-03
2,Não informado,48,8.971963,2026-03


### Número de mensagens inbound recebidas

### Número de mensagens inbound recebidas por ano-mês

### Número de mensagens inbound recebidas por dia da semana e hora-turno

### Número de Atendimentos total

### Número de Atendimentos por ano-mês (V)

In [248]:
def query_volumetria_dimensoes():
    where_clause = montar_filtros_sql("empresa_plano_saude_paciente")

    query = f"""
    WITH

    map_atendimento AS (
      SELECT 'Atendimento Médico UBS' AS descricao_atendimento, 'medico' AS categoria_atendimento UNION ALL
      SELECT 'Atendimento Médico UBS Baraúna' AS descricao_atendimento, 'medico' AS categoria_atendimento UNION ALL
      SELECT 'Consulta Covid MFC' AS descricao_atendimento, 'medico' AS categoria_atendimento UNION ALL
      SELECT 'Consulta de Queixa/Conduta MED' AS descricao_atendimento, 'medico' AS categoria_atendimento UNION ALL
      SELECT 'Consulta MFC' AS descricao_atendimento, 'medico' AS categoria_atendimento UNION ALL
      SELECT 'Consulta Programada MFC' AS descricao_atendimento, 'medico' AS categoria_atendimento UNION ALL
      SELECT 'Conversa Médica Linhas de Cuidado' AS descricao_atendimento, 'medico' AS categoria_atendimento UNION ALL
      SELECT 'MED - Administrativo (receitas, exames)' AS descricao_atendimento, 'medico' AS categoria_atendimento UNION ALL
      SELECT 'MED - Continuidade de cuidado' AS descricao_atendimento, 'medico' AS categoria_atendimento UNION ALL
      SELECT 'MED - Demanda espontânea' AS descricao_atendimento, 'medico' AS categoria_atendimento UNION ALL
      SELECT 'MED - Primeira consulta' AS descricao_atendimento, 'medico' AS categoria_atendimento UNION ALL
      SELECT 'Primeira Consulta MFC' AS descricao_atendimento, 'medico' AS categoria_atendimento UNION ALL
      SELECT 'Prescrição de Exame' AS descricao_atendimento, 'medico' AS categoria_atendimento UNION ALL
      SELECT 'Prescrição de Medicamento' AS descricao_atendimento, 'medico' AS categoria_atendimento UNION ALL

      SELECT 'Acolhimento de Saúde Mental' AS descricao_atendimento, 'psicoterapeuta' AS categoria_atendimento UNION ALL
      SELECT 'Acolhimento Psicológico' AS descricao_atendimento, 'psicoterapeuta' AS categoria_atendimento UNION ALL
      SELECT 'Acolhimento Saúde Mental' AS descricao_atendimento, 'psicoterapeuta' AS categoria_atendimento UNION ALL
      SELECT 'Consulta Multi' AS descricao_atendimento, 'psicoterapeuta' AS categoria_atendimento UNION ALL

      SELECT 'Acompanhamento de Saúde' AS descricao_atendimento, 'guardia' AS categoria_atendimento UNION ALL
      SELECT 'Atendimento de Saúde UBS Baraúna' AS descricao_atendimento, 'guardia' AS categoria_atendimento UNION ALL
      SELECT 'Avaliação/Orientação de Saúde' AS descricao_atendimento, 'guardia' AS categoria_atendimento UNION ALL
      SELECT 'Consulta de Queixa/Conduta ENF' AS descricao_atendimento, 'guardia' AS categoria_atendimento UNION ALL
      SELECT 'Consulta Programada' AS descricao_atendimento, 'guardia' AS categoria_atendimento UNION ALL
      SELECT 'Consulta Programada ENF' AS descricao_atendimento, 'guardia' AS categoria_atendimento UNION ALL
      SELECT 'Coordenação do cuidado' AS descricao_atendimento, 'guardia' AS categoria_atendimento UNION ALL
      SELECT 'Demanda Livre' AS descricao_atendimento, 'guardia' AS categoria_atendimento UNION ALL
      SELECT 'Consulta Covid Enfermagem' AS descricao_atendimento, 'guardia' AS categoria_atendimento UNION ALL
      SELECT 'ENF - Administrativo' AS descricao_atendimento, 'guardia' AS categoria_atendimento UNION ALL
      SELECT 'ENF - Administrativo / Navegação' AS descricao_atendimento, 'guardia' AS categoria_atendimento UNION ALL
      SELECT 'ENF - Busca ativa (agudo)' AS descricao_atendimento, 'guardia' AS categoria_atendimento UNION ALL
      SELECT 'ENF - Busca ativa (crônico)' AS descricao_atendimento, 'guardia' AS categoria_atendimento UNION ALL
      SELECT 'ENF - Busca ativa (navegação)' AS descricao_atendimento, 'guardia' AS categoria_atendimento UNION ALL
      SELECT 'ENF - Busca ativa (outros)' AS descricao_atendimento, 'guardia' AS categoria_atendimento UNION ALL
      SELECT 'ENF - Primeira Consulta' AS descricao_atendimento, 'guardia' AS categoria_atendimento UNION ALL
      SELECT 'ENF - Receptivo (agudo)' AS descricao_atendimento, 'guardia' AS categoria_atendimento UNION ALL
      SELECT 'ENF - Receptivo (crônico)' AS descricao_atendimento, 'guardia' AS categoria_atendimento UNION ALL
      SELECT 'ENF - Receptivo (eletivo)' AS descricao_atendimento, 'guardia' AS categoria_atendimento UNION ALL
      SELECT 'ENF - Receptivo (navegação)' AS descricao_atendimento, 'guardia' AS categoria_atendimento UNION ALL
      SELECT 'Jornadas de Cuidado' AS descricao_atendimento, 'guardia' AS categoria_atendimento UNION ALL
      SELECT 'Primeira Consulta' AS descricao_atendimento, 'guardia' AS categoria_atendimento UNION ALL
      SELECT 'Primeira Conversa' AS descricao_atendimento, 'guardia' AS categoria_atendimento UNION ALL

      SELECT 'felipe lobato' AS descricao_atendimento, 'Outro' AS categoria_atendimento UNION ALL
      SELECT 'Complemento prontuário anterior' AS descricao_atendimento, 'Outro' AS categoria_atendimento
    ),

    atendimentos_raw AS (
      SELECT
        g.registro_atendimento,
        g.id_atendimento,
        g.id_paciente,
        DATE(g.data_inicio_atendimento) AS data_atendimento,
        DATE_FORMAT(DATE(g.data_inicio_atendimento), '%%Y-%%m-01') AS mes,
        g.data_inicio_atendimento AS data_inicio_atendimento_ts,
        g.id_profissional,
        g.empresa_plano_saude_paciente,
        g.descricao_atendimento,
        COALESCE(m.categoria_atendimento, 'Nao mapeado') AS categoria_atendimento
      FROM communicare.geral g
      LEFT JOIN map_atendimento m
        ON g.descricao_atendimento = m.descricao_atendimento
      WHERE g.data_inicio_atendimento IS NOT NULL
        AND g.chegada IS NOT NULL
        AND (
          g.nome_paciente IS NULL
          OR INSTR(LOWER(g.nome_paciente), 'test') = 0
        )
        AND g.registro_atendimento IS NOT NULL
    ),

    atendimentos AS (
      SELECT
        registro_atendimento,
        id_atendimento,
        id_paciente,
        data_atendimento,
        mes,
        id_profissional,
        empresa_plano_saude_paciente,
        descricao_atendimento,
        categoria_atendimento
      FROM (
        SELECT
          a.*,
          ROW_NUMBER() OVER (
            PARTITION BY a.registro_atendimento
            ORDER BY a.data_inicio_atendimento_ts DESC, a.id_atendimento DESC
          ) AS rn
        FROM atendimentos_raw a
      ) t
      WHERE rn = 1
    ),

    agregado AS (
      SELECT
        CURRENT_TIMESTAMP AS data_atualizacao,
        mes,
        id_profissional,
        empresa_plano_saude_paciente,
        descricao_atendimento,
        categoria_atendimento,
        COUNT(*) AS qtd_atendimentos,
        SUM(CASE WHEN id_paciente IS NOT NULL THEN 1 ELSE 0 END) AS pacientes_atendidos_total,
        COUNT(DISTINCT id_paciente) AS pacientes_atendidos_unicos
      FROM atendimentos
      GROUP BY mes, id_profissional, empresa_plano_saude_paciente, descricao_atendimento, categoria_atendimento
    )

    SELECT *
    FROM agregado
    {where_clause}
    ORDER BY mes, empresa_plano_saude_paciente, categoria_atendimento, descricao_atendimento, id_profissional
    """
    return query

In [249]:
def preparar_volumetria_resumo_mensal(df, col_valor="qtd_atendimentos"):
    base = df.copy()

    base["mes"] = pd.to_datetime(base["mes"], errors="coerce")
    base[col_valor] = pd.to_numeric(base[col_valor], errors="coerce").fillna(0)

    resumo = (
        base.dropna(subset=["mes"])
        .groupby("mes", as_index=False)[col_valor]
        .sum()
        .rename(columns={col_valor: "volume"})
        .sort_values("mes")
        .reset_index(drop=True)
    )

    resumo["competencia"] = resumo["mes"].dt.to_period("M").astype(str)

    media_mensal = resumo["volume"].mean()
    maior_valor = resumo["volume"].max()
    menor_valor = resumo["volume"].min()

    resumo["variacao_percentual"] = resumo["volume"].pct_change() * 100
    resumo["indice"] = range(len(resumo))

    if len(resumo) >= 2:
        slope, intercept, *_ = linregress(resumo["indice"], resumo["volume"])
        resumo["tendencia"] = intercept + slope * resumo["indice"]
    else:
        resumo["tendencia"] = resumo["volume"]

    resumo["media_movel_6m"] = (
        resumo["volume"]
        .rolling(window=6, min_periods=1)
        .mean()
    )

    resumo["media_mensal"] = media_mensal
    resumo["maior_valor"] = maior_valor
    resumo["menor_valor"] = menor_valor

    resumo["texto_valor"] = resumo["volume"].apply(
        lambda v: f"{v:,.0f}".replace(",", ".")
    )

    return resumo

In [250]:
query_teste = query_volumetria_dimensoes()
df_teste = executar_query(query_teste, engine)
df_plot_teste = preparar_volumetria_resumo_mensal(df_teste)
display(df_teste.head())
display(df_plot_teste)
fig1 = grafico_volumetria_mensal(df_plot=df_plot_teste,detalhe="nenhum",titulo="Atendimentos por competência - teste manual")
fig1.show()

,data_atualizacao,mes,id_profissional,empresa_plano_saude_paciente,descricao_atendimento,categoria_atendimento,qtd_atendimentos,pacientes_atendidos_total,pacientes_atendidos_unicos
0,2026-03-20 18:17:05,2021-02-01,628,None,Coordenação do cuidado,guardia,8,8,6
1,2026-03-20 18:17:05,2021-02-01,483,None,Primeira Consulta,guardia,5,5,5
2,2026-03-20 18:17:05,2021-02-01,483,Ana Health,Consulta Programada,guardia,1,1,1
3,2026-03-20 18:17:05,2021-02-01,629,Ana Health,Demanda Livre,guardia,2,2,1
4,2026-03-20 18:17:05,2021-02-01,628,Oli Health,Coordenação do cuidado,guardia,10,10,10


,mes,volume,competencia,variacao_percentual,indice,tendencia,media_movel_6m,media_mensal,maior_valor,menor_valor,texto_valor
0,2021-02-01,41,2021-02,NaN,0,74.459293,41.000000,137.564516,547,11,41
1,2021-03-01,46,2021-03,12.195122,1,76.528317,43.500000,137.564516,547,11,46
2,2021-04-01,11,2021-04,-76.086957,2,78.597341,32.666667,137.564516,547,11,11
3,2021-05-01,37,2021-05,236.363636,3,80.666364,33.750000,137.564516,547,11,37
4,2021-06-01,39,2021-06,5.405405,4,82.735388,34.800000,137.564516,547,11,39
...,...,...,...,...,...,...,...,...,...,...,...
57,2025-11-01,113,2025-11,28.409091,57,192.393644,105.333333,137.564516,547,11,113
58,2025-12-01,94,2025-12,-16.814159,58,194.462668,101.666667,137.564516,547,11,94
59,2026-01-01,425,2026-01,352.127660,59,196.531691,157.166667,137.564516,547,11,425
60,2026-02-01,547,2026-02,28.705882,60,198.600715,233.166667,137.564516,547,11,547


### Número de Atendimento por dia da semana e hora-turno (V)

In [251]:
def query_volumetria_dia_semana_hora():
    where_clause = montar_filtros_sql("empresa_plano_saude_paciente")

    query = f"""
    WITH

    map_atendimento AS (
      SELECT 'Atendimento Médico UBS' AS descricao_atendimento, 'medico' AS categoria_atendimento UNION ALL
      SELECT 'Atendimento Médico UBS Baraúna' AS descricao_atendimento, 'medico' AS categoria_atendimento UNION ALL
      SELECT 'Consulta Covid MFC' AS descricao_atendimento, 'medico' AS categoria_atendimento UNION ALL
      SELECT 'Consulta de Queixa/Conduta MED' AS descricao_atendimento, 'medico' AS categoria_atendimento UNION ALL
      SELECT 'Consulta MFC' AS descricao_atendimento, 'medico' AS categoria_atendimento UNION ALL
      SELECT 'Consulta Programada MFC' AS descricao_atendimento, 'medico' AS categoria_atendimento UNION ALL
      SELECT 'Conversa Médica Linhas de Cuidado' AS descricao_atendimento, 'medico' AS categoria_atendimento UNION ALL
      SELECT 'MED - Administrativo (receitas, exames)' AS descricao_atendimento, 'medico' AS categoria_atendimento UNION ALL
      SELECT 'MED - Continuidade de cuidado' AS descricao_atendimento, 'medico' AS categoria_atendimento UNION ALL
      SELECT 'MED - Demanda espontânea' AS descricao_atendimento, 'medico' AS categoria_atendimento UNION ALL
      SELECT 'MED - Primeira consulta' AS descricao_atendimento, 'medico' AS categoria_atendimento UNION ALL
      SELECT 'Primeira Consulta MFC' AS descricao_atendimento, 'medico' AS categoria_atendimento UNION ALL
      SELECT 'Prescrição de Exame' AS descricao_atendimento, 'medico' AS categoria_atendimento UNION ALL
      SELECT 'Prescrição de Medicamento' AS descricao_atendimento, 'medico' AS categoria_atendimento UNION ALL

      SELECT 'Acolhimento de Saúde Mental' AS descricao_atendimento, 'psicoterapeuta' AS categoria_atendimento UNION ALL
      SELECT 'Acolhimento Psicológico' AS descricao_atendimento, 'psicoterapeuta' AS categoria_atendimento UNION ALL
      SELECT 'Acolhimento Saúde Mental' AS descricao_atendimento, 'psicoterapeuta' AS categoria_atendimento UNION ALL
      SELECT 'Consulta Multi' AS descricao_atendimento, 'psicoterapeuta' AS categoria_atendimento UNION ALL

      SELECT 'Acompanhamento de Saúde' AS descricao_atendimento, 'guardia' AS categoria_atendimento UNION ALL
      SELECT 'Atendimento de Saúde UBS Baraúna' AS descricao_atendimento, 'guardia' AS categoria_atendimento UNION ALL
      SELECT 'Avaliação/Orientação de Saúde' AS descricao_atendimento, 'guardia' AS categoria_atendimento UNION ALL
      SELECT 'Consulta de Queixa/Conduta ENF' AS descricao_atendimento, 'guardia' AS categoria_atendimento UNION ALL
      SELECT 'Consulta Programada' AS descricao_atendimento, 'guardia' AS categoria_atendimento UNION ALL
      SELECT 'Consulta Programada ENF' AS descricao_atendimento, 'guardia' AS categoria_atendimento UNION ALL
      SELECT 'Coordenação do cuidado' AS descricao_atendimento, 'guardia' AS categoria_atendimento UNION ALL
      SELECT 'Demanda Livre' AS descricao_atendimento, 'guardia' AS categoria_atendimento UNION ALL
      SELECT 'Consulta Covid Enfermagem' AS descricao_atendimento, 'guardia' AS categoria_atendimento UNION ALL
      SELECT 'ENF - Administrativo' AS descricao_atendimento, 'guardia' AS categoria_atendimento UNION ALL
      SELECT 'ENF - Administrativo / Navegação' AS descricao_atendimento, 'guardia' AS categoria_atendimento UNION ALL
      SELECT 'ENF - Busca ativa (agudo)' AS descricao_atendimento, 'guardia' AS categoria_atendimento UNION ALL
      SELECT 'ENF - Busca ativa (crônico)' AS descricao_atendimento, 'guardia' AS categoria_atendimento UNION ALL
      SELECT 'ENF - Busca ativa (navegação)' AS descricao_atendimento, 'guardia' AS categoria_atendimento UNION ALL
      SELECT 'ENF - Busca ativa (outros)' AS descricao_atendimento, 'guardia' AS categoria_atendimento UNION ALL
      SELECT 'ENF - Primeira Consulta' AS descricao_atendimento, 'guardia' AS categoria_atendimento UNION ALL
      SELECT 'ENF - Receptivo (agudo)' AS descricao_atendimento, 'guardia' AS categoria_atendimento UNION ALL
      SELECT 'ENF - Receptivo (crônico)' AS descricao_atendimento, 'guardia' AS categoria_atendimento UNION ALL
      SELECT 'ENF - Receptivo (eletivo)' AS descricao_atendimento, 'guardia' AS categoria_atendimento UNION ALL
      SELECT 'ENF - Receptivo (navegação)' AS descricao_atendimento, 'guardia' AS categoria_atendimento UNION ALL
      SELECT 'Jornadas de Cuidado' AS descricao_atendimento, 'guardia' AS categoria_atendimento UNION ALL
      SELECT 'Primeira Consulta' AS descricao_atendimento, 'guardia' AS categoria_atendimento UNION ALL
      SELECT 'Primeira Conversa' AS descricao_atendimento, 'guardia' AS categoria_atendimento UNION ALL

      SELECT 'felipe lobato' AS descricao_atendimento, 'Outro' AS categoria_atendimento UNION ALL
      SELECT 'Complemento prontuário anterior' AS descricao_atendimento, 'Outro' AS categoria_atendimento
    ),

    atendimentos_raw AS (
      SELECT
        g.registro_atendimento,
        g.id_atendimento,
        g.id_paciente,
        DATE(g.data_inicio_atendimento) AS data_atendimento,
        DATE_FORMAT(DATE(g.data_inicio_atendimento), '%%Y-%%m-01') AS mes,
        g.data_inicio_atendimento AS data_inicio_atendimento_ts,
        HOUR(g.data_inicio_atendimento) AS hora_dia,
        DAYOFWEEK(g.data_inicio_atendimento) AS dia_semana_mysql,
        CASE DAYOFWEEK(g.data_inicio_atendimento)
          WHEN 1 THEN 0
          WHEN 2 THEN 1
          WHEN 3 THEN 2
          WHEN 4 THEN 3
          WHEN 5 THEN 4
          WHEN 6 THEN 5
          WHEN 7 THEN 6
        END AS dia_semana_num,
        CASE DAYOFWEEK(g.data_inicio_atendimento)
          WHEN 1 THEN 'Sun'
          WHEN 2 THEN 'Mon'
          WHEN 3 THEN 'Tue'
          WHEN 4 THEN 'Wed'
          WHEN 5 THEN 'Thu'
          WHEN 6 THEN 'Fri'
          WHEN 7 THEN 'Sat'
        END AS dia_semana,
        g.id_profissional,
        g.empresa_plano_saude_paciente,
        g.descricao_atendimento,
        COALESCE(m.categoria_atendimento, 'Nao mapeado') AS categoria_atendimento
      FROM communicare.geral g
      LEFT JOIN map_atendimento m
        ON g.descricao_atendimento = m.descricao_atendimento
      WHERE g.data_inicio_atendimento IS NOT NULL
        AND g.chegada IS NOT NULL
        AND (
          g.nome_paciente IS NULL
          OR INSTR(LOWER(g.nome_paciente), 'test') = 0
        )
        AND g.registro_atendimento IS NOT NULL
    ),

    atendimentos AS (
      SELECT
        registro_atendimento,
        id_atendimento,
        id_paciente,
        data_atendimento,
        mes,
        hora_dia,
        dia_semana_num,
        dia_semana,
        id_profissional,
        empresa_plano_saude_paciente,
        descricao_atendimento,
        categoria_atendimento
      FROM (
        SELECT
          a.*,
          ROW_NUMBER() OVER (
            PARTITION BY a.registro_atendimento
            ORDER BY a.data_inicio_atendimento_ts DESC, a.id_atendimento DESC
          ) AS rn
        FROM atendimentos_raw a
      ) t
      WHERE rn = 1
    ),

    agregado AS (
      SELECT
        CURRENT_TIMESTAMP AS data_atualizacao,
        mes,
        dia_semana_num,
        dia_semana,
        hora_dia,
        id_profissional,
        empresa_plano_saude_paciente,
        descricao_atendimento,
        categoria_atendimento,
        COUNT(*) AS qtd_atendimentos
      FROM atendimentos
      GROUP BY
        mes,
        dia_semana_num,
        dia_semana,
        hora_dia,
        id_profissional,
        empresa_plano_saude_paciente,
        descricao_atendimento,
        categoria_atendimento
    )

    SELECT *
    FROM agregado
    {where_clause}
    ORDER BY mes, dia_semana_num, hora_dia, empresa_plano_saude_paciente
    """
    return query

In [252]:
def preparar_heatmap_dia_semana_hora(df):
    base = df.copy()

    base["hora_dia"] = pd.to_numeric(base["hora_dia"], errors="coerce")
    base["qtd_atendimentos"] = pd.to_numeric(base["qtd_atendimentos"], errors="coerce").fillna(0)
    base["dia_semana_num"] = pd.to_numeric(base["dia_semana_num"], errors="coerce")

    ordem_dias = ["Sun", "Mon", "Tue", "Wed", "Thu", "Fri", "Sat"]
    horas_ordem = list(range(24))

    resumo = (
        base.dropna(subset=["dia_semana", "hora_dia"])
        .groupby(["dia_semana_num", "dia_semana", "hora_dia"], as_index=False)["qtd_atendimentos"]
        .sum()
    )

    total_atendimentos = resumo["qtd_atendimentos"].sum()
    resumo["proporcao_total"] = 0 if total_atendimentos == 0 else resumo["qtd_atendimentos"] / total_atendimentos

    resumo["dia_semana"] = pd.Categorical(
        resumo["dia_semana"],
        categories=ordem_dias,
        ordered=True
    )

    resumo = resumo.sort_values(["dia_semana_num", "hora_dia"]).reset_index(drop=True)

    matriz = (
        resumo.pivot(index="hora_dia", columns="dia_semana", values="proporcao_total")
        .reindex(index=horas_ordem, columns=ordem_dias)
        .fillna(0)
    )

    barras_topo = (
        resumo.groupby("dia_semana", as_index=False)["proporcao_total"]
        .sum()
    )
    barras_topo["dia_semana"] = pd.Categorical(
        barras_topo["dia_semana"],
        categories=ordem_dias,
        ordered=True
    )
    barras_topo = barras_topo.sort_values("dia_semana").reset_index(drop=True)

    barras_lado = (
        resumo.groupby("hora_dia", as_index=False)["proporcao_total"]
        .sum()
        .sort_values("hora_dia")
        .reset_index(drop=True)
    )

    return {
        "resumo": resumo,
        "matriz": matriz,
        "barras_topo": barras_topo,
        "barras_lado": barras_lado,
        "total_atendimentos": total_atendimentos
    }

In [253]:
query_teste = query_volumetria_dia_semana_hora()
df_teste = executar_query(query_teste, engine)

df_plot_teste = preparar_heatmap_dia_semana_hora(df_teste)

fig1 = grafico_heatmap_dia_semana_hora(
    df_plot=df_plot_teste,
    titulo="Distribuição de atendimentos por dia da semana e hora - teste"
)

fig1.show()

C:\Users\GREG\AppData\Local\Temp\ipykernel_34044\265815051.py:35: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  resumo.groupby("dia_semana", as_index=False)["proporcao_total"]


### Número de motivos de atendimento por Categoria CIAP

### Número de motivos de atendimento por CIAP

### Número de atividades de enfermagem

### Número de atividades de enfermagem por ano-mês

### Número de atividades de enfermagem por status

### Número de atividades de enfermagem por mês por status

### Número de atividades de enfermagem por tipo

### Número de atividades de enfermagem por responsável

### Número de atividades de enfermagem por mês por responsável

### Número de captados total

### Número de captados total por ano-mês

### Número e percentual de atendimentos com soap (e quebras) preenchido

### Número e percentual de atendimentos com soap (e quebras) preenchido por ano-mês

### Número e percentual de atendimentos com soap (e quebras) preenchido por categoria profissional

### Número e percentual de atendimentos com soap (e quebras) preenchido por profissional

### Taxa de retorno médico (janela móvel 6m) média

### Taxa de retorno médico (janela móvel 6m) por ano-mês

### Taxa de resposta no WhatsApp da mensagem outbound

### Taxa de resposta no WhatsApp da mensagem outbound por ano-mês

### Taxa de resposta no WhatsApp da mensagem outbound por template

### Tempo de resposta

### Tempo de resposta estatísticas

### Tempo de resposta médio

### Tempo de resposta médio

### Engajamento

### Número de pacientes por condição

## Catálogo de Indicadores

In [254]:
catalogo_indicadores = {
    "demografia_ativos": {
        "ordem": 1,
        "nome_exibicao": "Beneficiários ativos por competência",
        "func_query": query_demografia_dimensoes,
        "func_preparacao": preparar_demografia_resumo_mensal,
        "func_grafico": grafico_demografia_ativos
    },
    "dinamica_beneficiarios": {
        "ordem": 2,
        "nome_exibicao": "Dinâmica mensal de beneficiários – Entradas e Saídas",
        "func_query": query_demografia_dimensoes,
        "func_preparacao": preparar_dinamica_beneficiarios,
        "func_grafico": grafico_dinamica_beneficiarios
    },
    "card_total_ativos": {
        "ordem": 3,
        "nome_exibicao": "Total de beneficiários ativos na última competência",
        "func_query": query_demografia_dimensoes,
        "func_preparacao": preparar_card_ativos_ultimo_mes,
        "func_grafico": grafico_card_total_ativos
    },
    "piramide_etaria": {
        "ordem": 4,
        "nome_exibicao": "Pirâmide etária por sexo no último mês",
        "func_query": query_demografia_dimensoes,
        "func_preparacao": preparar_piramide_etaria_ultimo_mes,
        "func_grafico": grafico_demografia_piramide_etaria
    },
    "faixa_etaria_ultimo_mes": {
        "ordem": 5,
        "nome_exibicao": "Beneficiários ativos por faixa etária na última competência",
        "func_query": query_demografia_dimensoes,
        "func_preparacao": preparar_faixa_etaria_ultimo_mes,
        "func_grafico": grafico_demografia_faixa_etaria
    },
    "sexo_ultimo_mes_setor": {
        "ordem": 6,
        "nome_exibicao": "Distribuição de beneficiários ativos por sexo na última competência",
        "func_query": query_demografia_dimensoes,
        "func_preparacao": preparar_sexo_ultimo_mes,
        "func_grafico": grafico_demografia_sexo_setor
    },
    "volumetria_atendimentos": {
        "ordem": 7,
        "nome_exibicao": "Atendimentos por competência",
        "func_query": query_volumetria_dimensoes,
        "func_preparacao": preparar_volumetria_resumo_mensal,
        "func_grafico": grafico_volumetria_mensal
    }
}

In [255]:
print(catalogo_indicadores.keys())
print(catalogo_indicadores["demografia_ativos"]["nome_exibicao"])

dict_keys(['demografia_ativos', 'dinamica_beneficiarios', 'card_total_ativos', 'piramide_etaria', 'faixa_etaria_ultimo_mes', 'sexo_ultimo_mes_setor', 'volumetria_atendimentos'])
Beneficiários ativos por competência


In [256]:
print(catalogo_indicadores.keys())
print(catalogo_indicadores["dinamica_beneficiarios"]["nome_exibicao"])

dict_keys(['demografia_ativos', 'dinamica_beneficiarios', 'card_total_ativos', 'piramide_etaria', 'faixa_etaria_ultimo_mes', 'sexo_ultimo_mes_setor', 'volumetria_atendimentos'])
Dinâmica mensal de beneficiários – Entradas e Saídas


## Processador

In [257]:
def processar_indicador_generico(
    nome_indicador,
    engine,
    nome_pasta="saida_indicadores",
    detalhe="completo",
    usar_titulo_catalogo=True
):
    if nome_indicador not in catalogo_indicadores:
        raise ValueError(f"Indicador '{nome_indicador}' não encontrado.")

    config = catalogo_indicadores[nome_indicador]

    query = config["func_query"]()
    df_bruto = executar_query(query, engine)
    df_plot = config["func_preparacao"](df_bruto)

    titulo = config["nome_exibicao"] if usar_titulo_catalogo else None

    print(f"Detalhe recebido em processar_indicador_generico: {detalhe}")
    print(f"Indicador: {nome_indicador}")

    if titulo is not None:
        fig = config["func_grafico"](
            df_plot,
            detalhe=detalhe,
            titulo=titulo
        )
    else:
        fig = config["func_grafico"](
            df_plot,
            detalhe=detalhe
        )

    caminho_csv_bruto = exportar_dataframe(
        df=df_bruto,
        nome_indicador=nome_indicador,
        nome_pasta=nome_pasta,
        tipo_df="bruto"
    )

    caminho_csv_plot = exportar_dataframe(
        df=df_plot,
        nome_indicador=nome_indicador,
        nome_pasta=nome_pasta,
        tipo_df="plot"
    )

    caminho_html = exportar_grafico_html(
        fig=fig,
        nome_indicador=nome_indicador,
        nome_pasta=nome_pasta
    )

    caminho_png = None
    try:
        caminho_png = exportar_grafico_png(
            fig=fig,
            nome_indicador=nome_indicador,
            nome_pasta=nome_pasta
        )
    except Exception:
        print(f"PNG não exportado para {nome_indicador}.")

    return {
        "nome": nome_indicador,
        "nome_exibicao": config["nome_exibicao"],
        "query": query,
        "df_bruto": df_bruto,
        "df_plot": df_plot,
        "fig": fig,
        "caminho_csv_bruto": caminho_csv_bruto,
        "caminho_csv_plot": caminho_csv_plot,
        "caminho_html": caminho_html,
        "caminho_png": caminho_png
    }

In [258]:
def executar_varios_indicadores(
    lista_indicadores,
    engine,
    nome_pasta="saida_indicadores",
    detalhe="completo",
    usar_titulo_catalogo=True
):
    resultados = {}

    for nome_indicador in lista_indicadores:
        print(f"Rodando indicador: {nome_indicador}")

        try:
            resultado = processar_indicador_generico(
                nome_indicador=nome_indicador,
                engine=engine,
                nome_pasta=nome_pasta,
                detalhe=detalhe,
                usar_titulo_catalogo=usar_titulo_catalogo
            )
            resultados[nome_indicador] = resultado
            print(f"Concluído: {nome_indicador}\n")

        except Exception as e:
            print(f"Erro no indicador {nome_indicador}: {e}\n")
            resultados[nome_indicador] = {
                "nome": nome_indicador,
                "erro": str(e)
            }

    return resultados

In [259]:
def executar_todos_indicadores(
    engine,
    nome_pasta="saida_indicadores",
    detalhe="completo",
    usar_titulo_catalogo=True
):
    lista_indicadores = list(catalogo_indicadores.keys())

    return executar_varios_indicadores(
        lista_indicadores=lista_indicadores,
        engine=engine,
        nome_pasta=nome_pasta,
        detalhe=detalhe,
        usar_titulo_catalogo=usar_titulo_catalogo
    )

In [260]:
#resultados_todos = executar_todos_indicadores(
#    engine=engine,
#    detalhe=detalhe_relatorio
#)

#print(resultados_todos.keys())

# Geração do Relatório 

In [261]:
# =========================
# Exibição do formulário
# =========================
display(
    widgets.VBox([
        dropdown_empresa,
        data_inicial_widget,
        data_final_widget,
        dropdown_detalhe,
        botao_confirmar,
        botao_rodar_indicadores,
        saida_filtros,
        saida_execucao
    ])
)

# 📊 Sugestão de Sequência para montagem da Apresentação

1. demografia_ativos_grafico_selecao_empresas_selecao_data.png
2. dinamica_beneficiarios_grafico_selecao_empresas_selecao_data.png
3. card_total_ativos_bruto_selecao_empresas_selecao_data.png
4. piramide_etaria_plot_selecao_empresas_selecao_data.png
5. faixa_etaria_ultimo_mes_grafico_selecao_empresas_selecao_data.png
6. sexo_ultimo_mes_setor_grafico_selecao_empresas_selecao_data.png
7. volumetria_atendimentos_selecao_empresas_selecao_data.png